1. Configuración Inicial y Metadatos

In [ ]:

"""
ANÁLISIS HÍBRIDO DE AUSENTISMO LABORAL - EQUIPO 15
Fecha: 25/09/2025
Versión: 2.0
Autor: Equipo de Análisis de Datos - Oriac Gimeno
Descripción: Análisis completo de factores que influyen en el ausentismo laboral
"""

2. Configuración de Entorno y Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import shapiro, spearmanr, kruskal, f_oneway, chi2_contingency, pointbiserialr, mannwhitneyu, ttest_ind, fisher_exact
import warnings
import os
from datetime import datetime

# Configuración de estilo y warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuración de rutas
class Config:
    DATA_PATH = r"C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\RRHH_220925_clean.parquet"
    OUTPUT_BASE = r"G:\Mi unidad\IT ACADEMY\Reskilling Data Analytics\Simulador Empresarial\Results"
    
    @property
    def OUTPUT_PATH(self):
        return os.path.join(self.OUTPUT_BASE, "Resultados_Analisis_Absentisme_i_benestar_laboral_220925_DEFINITIVO_2")

config = Config()

3. Clases de Utilidades para Análisis Estadístico

In [ ]:
class StatisticalAnalyzer:
    """Clase para realizar análisis estadísticos de manera estructurada"""
    
    @staticmethod
    def clasificar_correlacion(corr):
        """Clasificar la fuerza de la correlación según criterios estándar"""
        abs_corr = abs(corr)
        if abs_corr < 0.1:
            return "Muy débil", 1
        elif abs_corr < 0.3:
            return "Débil", 2
        elif abs_corr < 0.5:
            return "Moderada", 3
        elif abs_corr < 0.7:
            return "Fuerte", 4
        else:
            return "Muy fuerte", 5
    
    @staticmethod
    def cohens_d(x, y):
        """Calcular tamaño del efecto Cohen's d"""
        nx, ny = len(x), len(y)
        dof = nx + ny - 2
        pooled_std = np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / dof)
        return (np.mean(x) - np.mean(y)) / pooled_std
    
    @staticmethod
    def clasificar_efecto_cohen(d):
        """Clasificar el tamaño del efecto de Cohen"""
        abs_d = abs(d)
        if abs_d < 0.2:
            return "Muy pequeño"
        elif abs_d < 0.5:
            return "Pequeño"
        elif abs_d < 0.8:
            return "Mediano"
        else:
            return "Grande"

class DataValidator:
    """Clase para validaciones de datos"""
    
    @staticmethod
    def verificar_variables(df, variables_requeridas):
        """Verificar que las variables requeridas existen en el DataFrame"""
        faltantes = [var for var in variables_requeridas if var not in df.columns]
        if faltantes:
            raise ValueError(f"Variables faltantes en el dataset: {faltantes}")
        print("✓ Todas las variables requeridas están presentes")
        
    @staticmethod
    def verificar_tamanio_muestral(grupo1, grupo2, min_tamano=5):
        """Verificar tamaño muestral mínimo para análisis"""
        if len(grupo1) < min_tamano or len(grupo2) < min_tamano:
            return False
        return True
    
    @staticmethod
    def verificar_binarias(df, binary_vars):
        """Verificar que las variables binarias contienen solo 0 y 1"""
        print("\n🔍 Validación de variables binarias:")
        for var in binary_vars:
            unique_vals = sorted(df[var].unique())
            if set(unique_vals) <= {0, 1}:
                print(f"  ✓ {var}: binaria correcta [valores: {unique_vals}]")
            else:
                print(f"  ⚠ {var}: NO es estrictamente binaria [valores: {unique_vals}]")
    
    @staticmethod
    def calidad_datos_report(df):
        """Generar reporte completo de calidad de datos"""
        print("\n📊 REPORTE DE CALIDAD DE DATOS:")
        print(f"  • Número de observaciones: {len(df):,}")
        print(f"  • Número de variables: {len(df.columns)}")
        print(f"  • Valores faltantes totales: {df.isnull().sum().sum()}")
        print(f"  • Filas duplicadas: {df.duplicated().sum()}")
        
        # Variables constantes
        constant_vars = [col for col in df.columns if df[col].nunique() == 1]
        if constant_vars:
            print(f"  ⚠ Variables constantes: {constant_vars}")
        else:
            print("  ✓ No hay variables constantes")
        
        # Valores faltantes por variable
        missing_per_var = df.isnull().sum()
        if missing_per_var.any():
            print("\n  🔍 Valores faltantes por variable:")
            for var, count in missing_per_var[missing_per_var > 0].items():
                print(f"    {var}: {count} ({count/len(df)*100:.2f}%)")
        else:
            print("  ✓ No hay valores faltantes")

class BinaryVariablesAnalyzerCorregido:
    """Analizador corregido para variables binarias con enfoques apropiados"""
    
    @staticmethod
    def analizar_binarias_corregido(df, binary_vars, target_cont, target_bin):
        """Análisis corregido para variables binarias usando enfoques apropiados"""
        print("\n🔬 ANÁLISIS CORREGIDO DE VARIABLES BINARIAS:")
        resultados_bin = []
        
        for var in binary_vars:
            if var not in df.columns:
                continue
                
            try:
                # Datos para grupos
                grupo_0 = df[df[var] == 0][target_cont].dropna()
                grupo_1 = df[df[var] == 1][target_cont].dropna()
                
                if len(grupo_0) < 5 or len(grupo_1) < 5:
                    print(f"  ⚠ {var}: Tamaño muestral insuficiente")
                    continue
                
                print(f"  🔍 Analizando {var}:")
                print(f"    - Grupo 0 (n={len(grupo_0)}): media={grupo_0.mean():.2f}")
                print(f"    - Grupo 1 (n={len(grupo_1)}): media={grupo_1.mean():.2f}")
                
                # ENFOQUE 1: Como variable categórica (comparación de medias)
                # ------------------------------------------------------------
                
                # Test de normalidad de los residuos (para decidir test paramétrico vs no paramétrico)
                normalidad_grupo0 = shapiro(grupo_0)[1] > 0.05 if len(grupo_0) >= 3 else False
                normalidad_grupo1 = shapiro(grupo_1)[1] > 0.05 if len(grupo_1) >= 3 else False
                varianzas_iguales = stats.levene(grupo_0, grupo_1)[1] > 0.05
                
                # Decidir test apropiado
                if normalidad_grupo0 and normalidad_grupo1 and varianzas_iguales:
                    # T-test para muestras independientes
                    t_stat, t_p = ttest_ind(grupo_0, grupo_1, equal_var=True)
                    test_utilizado = "T-test"
                    estadistico = t_stat
                    p_valor = t_p
                else:
                    # Mann-Whitney U test (no paramétrico)
                    mw_stat, mw_p = mannwhitneyu(grupo_0, grupo_1, alternative='two-sided')
                    test_utilizado = "Mann-Whitney"
                    estadistico = mw_stat
                    p_valor = mw_p
                
                # Tamaño del efecto (Cohen's d)
                d_effect = StatisticalAnalyzer.cohens_d(grupo_0, grupo_1)
                tamaño_efecto = StatisticalAnalyzer.clasificar_efecto_cohen(d_effect)
                
                # ENFOQUE 2: Como variable numérica (correlación punto-biserial) - CORREGIDO
                # -------------------------------------------------------------
                try:
                    # Filtrar datos completos para correlación
                    datos_completos = df[[var, target_cont]].dropna()
                    if len(datos_completos) >= 3:
                        pointbiserial_corr, pointbiserial_p = pointbiserialr(datos_completos[var], datos_completos[target_cont])
                        fuerza_pb, _ = StatisticalAnalyzer.clasificar_correlacion(pointbiserial_corr)
                    else:
                        pointbiserial_corr, pointbiserial_p, fuerza_pb = np.nan, np.nan, "N/A"
                except Exception as e:
                    print(f"    ⚠ Error en correlación punto-biserial para {var}: {e}")
                    pointbiserial_corr, pointbiserial_p, fuerza_pb = np.nan, np.nan, "Error"
                
                # ENFOQUE 3: Asociación con variable binaria objetivo (Odds Ratio) - VERSIÓN CORREGIDA
                # ---------------------------------------------------------------
                try:
                    tabla_contingencia = pd.crosstab(df[var], df[target_bin])
                    
                    # Asegurar que la tabla sea 2x2 y esté ordenada correctamente
                    if tabla_contingencia.shape == (2, 2):
                        # Reindexar para asegurar orden 0,1
                        tabla_contingencia = tabla_contingencia.reindex(index=[0, 1], columns=[0, 1])
                        
                        # Extraer valores en orden correcto
                        a = tabla_contingencia.iloc[0, 0]  # var=0, target=0
                        b = tabla_contingencia.iloc[0, 1]  # var=0, target=1  
                        c = tabla_contingencia.iloc[1, 0]  # var=1, target=0
                        d = tabla_contingencia.iloc[1, 1]  # var=1, target=1
                        
                        print(f"    - Tabla contingencia corregida: [{a}, {b}; {c}, {d}]")
                        
                        # CÁLCULOS CORRECTOS DE RISK RATIO
                        riesgo_grupo0 = b / (a + b) if (a + b) > 0 else 0
                        riesgo_grupo1 = d / (c + d) if (c + d) > 0 else 0
                        risk_ratio = riesgo_grupo1 / riesgo_grupo0 if riesgo_grupo0 > 0 else np.nan
                        
                        # CÁLCULOS CORRECTOS DE ODDS RATIO
                        odds_grupo0 = b / a if a > 0 else np.nan
                        odds_grupo1 = d / c if c > 0 else np.nan
                        odds_ratio = odds_grupo1 / odds_grupo0 if not np.isnan(odds_grupo1) and not np.isnan(odds_grupo0) else np.nan
                        
                        # Test exacto de Fisher
                        odds_ratio_fisher, fisher_p = fisher_exact(tabla_contingencia)
                        
                        print(f"    - Risk Ratio CORREGIDO: {risk_ratio:.3f}")
                        print(f"    - Odds Ratio CORREGIDO: {odds_ratio:.3f}")
                        
                    else:
                        odds_ratio, odds_ratio_fisher, fisher_p, risk_ratio = np.nan, np.nan, np.nan, np.nan
                        print(f"    ⚠ Tabla de contingencia no es 2x2: {tabla_contingencia.shape}")
                        
                except Exception as e:
                    odds_ratio, odds_ratio_fisher, fisher_p, risk_ratio = np.nan, np.nan, np.nan, np.nan
                    print(f"    ⚠ Error en tabla de contingencia para {var}: {e}")
                
                # Resultados consolidados
                resultados_bin.append({
                    'Variable': var,
                    'Test_Utilizado': test_utilizado,
                    'Estadistico_Test': round(estadistico, 3),
                    'p_valor': round(p_valor, 6),
                    'Significativa': p_valor < 0.05,
                    'Media_Grupo0': round(grupo_0.mean(), 3),
                    'Media_Grupo1': round(grupo_1.mean(), 3),
                    'Diferencia_Medias': round(grupo_1.mean() - grupo_0.mean(), 3),
                    'Cohens_d': round(d_effect, 3),
                    'Tamaño_Efecto': tamaño_efecto,
                    'PointBiserial_Corr': round(pointbiserial_corr, 3) if not np.isnan(pointbiserial_corr) else np.nan,
                    'PointBiserial_p': round(pointbiserial_p, 6) if not np.isnan(pointbiserial_p) else np.nan,
                    'Fuerza_PointBiserial': fuerza_pb,
                    'Odds_Ratio': round(odds_ratio, 3) if not np.isnan(odds_ratio) else np.nan,
                    'Fisher_p': round(fisher_p, 6) if not np.isnan(fisher_p) else np.nan,
                    'Risk_Ratio': round(risk_ratio, 3) if not np.isnan(risk_ratio) else np.nan,
                    'Interpretacion_Direccion': "AUMENTA" if (grupo_1.mean() > grupo_0.mean()) else "DISMINUYE"
                })
                
                print(f"    ✓ Análisis completado: p={p_valor:.6f}, Cohen's d={d_effect:.3f}")
                
            except Exception as e:
                print(f"  ✗ Error analizando {var}: {e}")
        
        return pd.DataFrame(resultados_bin)
    
    @staticmethod
    def interpretar_resultados_binarios(df_results_bin):
        """Interpretación mejorada de resultados binarios"""
        if df_results_bin.empty:
            print("  No hay resultados para interpretar")
            return
            
        print("\n📋 INTERPRETACIÓN CORREGIDA - VARIABLES BINARIAS:")
        print("="*60)
        
        for _, row in df_results_bin.iterrows():
            if row['Significativa']:
                # Interpretación basada en diferencia de medias
                efecto = row['Interpretacion_Direccion']
                print(f"\n  🎯 {row['Variable']}:")
                print(f"     • Efecto: {efecto} el ausentismo ({row['Test_Utilizado']}, p={row['p_valor']:.4f})")
                print(f"     • Diferencia: {row['Media_Grupo1']:.2f} vs {row['Media_Grupo0']:.2f} horas")
                print(f"     • Tamaño efecto: Cohen's d = {row['Cohens_d']:.2f} ({row['Tamaño_Efecto']})")
                
                # Interpretación punto-biserial si está disponible
                if not np.isnan(row['PointBiserial_Corr']):
                    print(f"     • Correlación punto-biserial: {row['PointBiserial_Corr']:.3f} ({row['Fuerza_PointBiserial']})")
                
                # Interpretación odds ratio si está disponible
                if not np.isnan(row['Odds_Ratio']):
                    interpretacion_or = "aumenta riesgo" if row['Odds_Ratio'] > 1 else "disminuye riesgo"
                    print(f"     • Odds Ratio: {row['Odds_Ratio']:.2f} ({interpretacion_or})")
                
                # Interpretación risk ratio si está disponible
                if not np.isnan(row['Risk_Ratio']):
                    interpretacion_rr = "aumenta riesgo" if row['Risk_Ratio'] > 1 else "disminuye riesgo"
                    print(f"     • Risk Ratio: {row['Risk_Ratio']:.2f} ({interpretacion_rr})")
            else:
                print(f"\n  📊 {row['Variable']}: No significativa (p={row['p_valor']:.4f})")

4. Carga y Validación de Datos

In [ ]:
print("="*80)
print("FASE 1: CARGA Y PREPARACIÓN DE DATOS")
print("="*80)

try:
    # Cargar datos
    df = pd.read_parquet(config.DATA_PATH)
    print(f"✓ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    
    # Definir variables con clasificación CORREGIDA
    VARIABLES_BINARIAS = ['Disciplinary_failure', 'Social_drinker', 'Social_smoker']

    # Variables personales (incluye Reason_absence_numeric como personal)
    VARIABLES_PERSONALES = [
        'Age', 'Son', 'Pet', 'Weight', 'Height', 
        'Body_mass_index', 'Education_numeric', 'Social_drinker', 'Social_smoker',
        'Reason_absence_numeric'  # CORREGIDO: es variable personal
    ]
    
    # Variables laborales
    VARIABLES_LABORALES = [
        'Transportation_expense', 'Distance_Residence_Work', 'Service_time',
        'Work_load_Average_day', 'Hit_target', 'Disciplinary_failure'
    ]
    
    # Variables temporales (EXCLUIDAS del análisis principal)
    VARIABLES_TEMPORALES = [
        'Month_absence', 'Day_week', 'Seasons', 'Reason_absence'  # Versiones categóricas con strings
    ]
    
    # Variables temporales numéricas (para análisis específico si se necesita)
    VARIABLES_TEMPORALES_NUMERICAS = [
        'Month_absence_order', 'Day_week_order', 'Seasons_order'
    ]
    
    # CORRECCIÓN: VARIABLES_NUMERICAS solo debe incluir variables verdaderamente numéricas
    VARIABLES_NUMERICAS = VARIABLES_PERSONALES + VARIABLES_LABORALES + VARIABLES_TEMPORALES_NUMERICAS

    VARIABLES_CATEGORICAS = ['Reason_absence', 'Month_absence', 'Day_week', 'Seasons', 'Education'] + VARIABLES_BINARIAS
    TARGET_CONTINUO = 'Absenteeism_hours'
    
    # Validar variables
    todas_variables = VARIABLES_NUMERICAS + VARIABLES_CATEGORICAS + [TARGET_CONTINUO]
    DataValidator.verificar_variables(df, todas_variables)
    
    # Validaciones específicas
    DataValidator.verificar_binarias(df, VARIABLES_BINARIAS)
    DataValidator.calidad_datos_report(df)
    
    # Preparar datos - CREAR 3 CATEGORÍAS DE AUSENTISMO
    df_clean = df.drop(columns=['ID'], errors='ignore')
    
    # Crear 3 categorías de ausentismo (33%, 33-66%, >66%)
    percentil_33 = df_clean[TARGET_CONTINUO].quantile(0.33)
    percentil_66 = df_clean[TARGET_CONTINUO].quantile(0.66)
    
    def categorizar_ausentismo(horas):
        if horas <= percentil_33:
            return 'Bajo'
        elif horas <= percentil_66:
            return 'Medio'
        else:
            return 'Alto'
    
    df_clean['Ausentismo_Categoria'] = df_clean[TARGET_CONTINUO].apply(categorizar_ausentismo)
    df_clean['Ausentismo_Alto'] = (df_clean[TARGET_CONTINUO] > percentil_66).astype(int)
    TARGET_BINARIO = 'Ausentismo_Alto'
    TARGET_CATEGORICO = 'Ausentismo_Categoria'
    
    print(f"✓ Datos preparados. Categorías de ausentismo:")
    print(f"  - Bajo: ≤ {percentil_33:.2f} horas (33% inferior)")
    print(f"  - Medio: {percentil_33:.2f} - {percentil_66:.2f} horas")
    print(f"  - Alto: > {percentil_66:.2f} horas (33% superior)")
    print(f"✓ Variables creadas: {TARGET_BINARIO} (binario), {TARGET_CATEGORICO} (categórico)")
    print(f"✓ Variables personales: {len(VARIABLES_PERSONALES)}")
    print(f"✓ Variables laborales: {len(VARIABLES_LABORALES)}")
    print(f"✓ Variables temporales (excluidas): {len(VARIABLES_TEMPORALES)}")
    print(f"✓ Variables numéricas totales: {len(VARIABLES_NUMERICAS)}")
    
    # Verificación de tipos de datos
    print(f"\n🔍 VERIFICACIÓN DE TIPOS DE DATOS EN VARIABLES_NUMERICAS:")
    for var in VARIABLES_NUMERICAS:
        if var in df_clean.columns:
            dtype = df_clean[var].dtype
            print(f"  ✓ {var}: {dtype}")
    
except Exception as e:
    print(f"✗ Error en carga de datos: {e}")
    raise

4. 5. ANÁLISIS DE CALIDAD DE DATOS

5. Análisis de Normalidad

In [ ]:
print("\n" + "="*80)
print("FASE 5: ANÁLISIS DE NORMALIDAD (Shapiro-Wilk)")
print("="*80)

def analizar_normalidad(df, variables):
    """Realizar test de normalidad Shapiro-Wilk para múltiples variables"""
    resultados = []
    
    # Filtrar solo variables numéricas
    variables_numericas_reales = [var for var in variables if pd.api.types.is_numeric_dtype(df[var])]
    variables_no_numericas = [var for var in variables if not pd.api.types.is_numeric_dtype(df[var])]
    
    if variables_no_numericas:
        print(f"⚠ Variables no numéricas excluidas del análisis de normalidad: {variables_no_numericas}")
    
    for var in variables_numericas_reales:
        data = df[var].dropna()
        if len(data) > 3:  # Mínimo requerido para Shapiro-Wilk
            try:
                stat, p_value = shapiro(data)
                resultados.append({
                    'Variable': var,
                    'Estadistico_Shapiro': round(stat, 6),
                    'p_value_Shapiro': round(p_value, 6),
                    'Es_Normal': p_value > 0.05,
                    'n': len(data)
                })
            except Exception as e:
                print(f"⚠ Error en test de normalidad para {var}: {e}")
        else:
            print(f"⚠ Variable {var} no tiene suficientes datos para análisis de normalidad (n={len(data)})")
    
    return pd.DataFrame(resultados)

# Crear lista CORREGIDA de variables para análisis de normalidad
# Solo incluir variables personales, laborales y el target (excluyendo variables temporales categóricas)
variables_para_normalidad = VARIABLES_PERSONALES + VARIABLES_LABORALES + [TARGET_CONTINUO]

# Asegurarnos de que Month_absence_order esté incluida si existe
if 'Month_absence_order' in df_clean.columns and 'Month_absence_order' not in variables_para_normalidad:
    variables_para_normalidad.append('Month_absence_order')

print("🔍 Variables incluidas en análisis de normalidad:")
print(f"  - Personales: {len(VARIABLES_PERSONALES)} variables")
print(f"  - Laborales: {len(VARIABLES_LABORALES)} variables")
print(f"  - Target: 1 variable")
print(f"  - Total: {len(variables_para_normalidad)} variables")

# Verificación adicional de tipos de datos
print(f"\n🔍 VERIFICACIÓN DE TIPOS DE DATOS PARA ANÁLISIS DE NORMALIDAD:")
variables_numericas_verificadas = []
variables_excluidas = []

for var in variables_para_normalidad:
    if var in df_clean.columns:
        if pd.api.types.is_numeric_dtype(df_clean[var]):
            n_missing = df_clean[var].isnull().sum()
            unique_vals = df_clean[var].dropna().unique()
            print(f"  ✓ {var}: numérica (tipo: {df_clean[var].dtype}, faltantes: {n_missing}, valores únicos: {len(unique_vals)})")
            variables_numericas_verificadas.append(var)
        else:
            print(f"  ✗ {var}: NO numérica (tipo: {df_clean[var].dtype}) - EXCLUIDA")
            variables_excluidas.append(var)
    else:
        print(f"  ⚠ {var}: no encontrada en el dataset")
        variables_excluidas.append(var)

print(f"\n📋 RESUMEN:")
print(f"  - Variables verificadas: {len(variables_numericas_verificadas)}")
print(f"  - Variables excluidas: {len(variables_excluidas)}")

# Ejecutar análisis solo con variables numéricas verificadas
df_normality = analizar_normalidad(df_clean, variables_numericas_verificadas)

if not df_normality.empty:
    print("\n📊 RESULTADOS DE NORMALIDAD (Shapiro-Wilk):")
    print(df_normality.to_string(index=False))
    
    # Resumen estadístico
    normales = df_normality['Es_Normal'].sum()
    total_vars = len(df_normality)
    print(f"\n📊 RESUMEN ESTADÍSTICO:")
    print(f"  - Variables analizadas: {total_vars}")
    print(f"  - Variables normales: {normales} ({normales/total_vars*100:.1f}%)")
    print(f"  - Variables no normales: {total_vars - normales} ({(total_vars - normales)/total_vars*100:.1f}%)")
    
    # Variables no normales (para referencia)
    variables_no_normales = df_normality[~df_normality['Es_Normal']]['Variable'].tolist()
    if variables_no_normales:
        print(f"  📋 Variables NO normales: {', '.join(variables_no_normales)}")
    
    # Implicaciones para el análisis
    print(f"\n💡 IMPLICACIONES ESTADÍSTICAS:")
    if normales == total_vars:
        print("  - TODAS las variables son normales → Se pueden usar tests PARAMÉTRICOS")
    elif normales == 0:
        print("  - NINGUNA variable es normal → Se usarán tests NO PARAMÉTRICOS")
        print("  - Tests a usar: Spearman, Mann-Whitney, Kruskal-Wallis")
    else:
        print("  - ALGUNAS variables no son normales → Se recomiendan tests NO PARAMÉTRICOS")
        print("  - Tests a usar: Spearman, Mann-Whitney, Kruskal-Wallis")
        
    print(f"\n🎯 DECISIÓN FINAL: Se usarán tests NO PARAMÉTRICOS para mayor robustez")
    
else:
    print("❌ No se pudo realizar el análisis de normalidad - no hay variables numéricas válidas")
    # Crear un DataFrame vacío con la estructura esperada para evitar errores posteriores
    df_normality = pd.DataFrame(columns=['Variable', 'Estadistico_Shapiro', 'p_value_Shapiro', 'Es_Normal', 'n'])

# Información adicional sobre variables temporales (para referencia)
print(f"\n📌 NOTA SOBRE VARIABLES TEMPORALES:")
variables_temporales_numericas = [var for var in VARIABLES_TEMPORALES if var in df_clean.columns and pd.api.types.is_numeric_dtype(df_clean[var])]
variables_temporales_categoricas = [var for var in VARIABLES_TEMPORALES if var in df_clean.columns and not pd.api.types.is_numeric_dtype(df_clean[var])]

if variables_temporales_numericas:
    print(f"  - Variables temporales numéricas (no incluidas en análisis principal): {variables_temporales_numericas}")
if variables_temporales_categoricas:
    print(f"  - Variables temporales categóricas: {variables_temporales_categoricas}")

6. Análisis de Variables Numéricas

In [ ]:
print("\n" + "="*80)
print("FASE 6: ANÁLISIS DE VARIABLES NUMÉRICAS")
print("="*80)

def analizar_variables_numericas(df, variables, target_cont, target_bin, df_normality):
    """Análisis completo para variables numéricas"""
    resultados = []
    analyzer = StatisticalAnalyzer()
    
    for i, var in enumerate(variables, 1):
        print(f"🔍 Analizando {i}/{len(variables)}: {var}")
        
        if var not in df.columns:
            print(f"  ⚠ Variable {var} no encontrada, saltando...")
            continue
            
        # Preparar datos para grupos
        data_alto = df[df[target_bin] == 1][var].dropna()
        data_bajo = df[df[target_bin] == 0][var].dropna()
        
        # Verificar tamaño muestral
        if not DataValidator.verificar_tamanio_muestral(data_alto, data_bajo):
            print(f"  ⚠ Tamaño muestral insuficiente para {var}, saltando...")
            continue
        
        try:
            # Correlación Spearman con manejo robusto de datos
            data_var = df[var].dropna()
            data_target = df[target_cont].dropna()
            
            # Alinear índices para evitar error de dimensiones
            common_idx = data_var.index.intersection(data_target.index)
            if len(common_idx) < 3:
                print(f"  ⚠ Datos insuficientes para correlación en {var}")
                continue
                
            spearman_corr, spearman_p = spearmanr(data_var.loc[common_idx], data_target.loc[common_idx])
            fuerza_spearman, fuerza_num = analyzer.clasificar_correlacion(spearman_corr)
            
            # Point-biserial
            pointbiserial_corr, pointbiserial_p = pointbiserialr(df[var].dropna(), df[target_bin].dropna())
            
            # Tests de diferencias
            mw_stat, mw_p = mannwhitneyu(data_alto, data_bajo, alternative='two-sided')
            t_stat, t_p = ttest_ind(data_alto, data_bajo, equal_var=False)
            
            # Tamaño del efecto
            d_effect = analyzer.cohens_d(data_alto, data_bajo)
            tamaño_efecto = analyzer.clasificar_efecto_cohen(d_effect)
            
            # Determinar test apropiado
            es_normal_var = df_normality[df_normality['Variable'] == var]['Es_Normal'].values[0]
            test_medias = "T-test" if es_normal_var else "Mann-Whitney"
            p_value_medias = t_p if es_normal_var else mw_p
            
            resultados.append({
                'Variable': var,
                'Correlacion_Spearman': round(spearman_corr, 6),
                'p_value_Spearman': round(spearman_p, 6),
                'Fuerza_Spearman': fuerza_spearman,
                'Significativa_Spearman': spearman_p < 0.05,
                'Cohens_d': round(d_effect, 6),
                'Tamaño_Efecto': tamaño_efecto,
                'Test_Medias': test_medias,
                'p_value_Medias': round(p_value_medias, 6),
                'Significativa_Medias': p_value_medias < 0.05,
                'Media_Alto': round(data_alto.mean(), 3),
                'Media_Bajo': round(data_bajo.mean(), 3),
                'Diferencia_Medias': round(data_alto.mean() - data_bajo.mean(), 3),
                'abs_corr': abs(spearman_corr),
                'abs_effect': abs(d_effect)
            })
            
        except Exception as e:
            print(f"  ⚠ Error en {var}: {e}")
    
    return pd.DataFrame(resultados)

# Ejecutar análisis
df_results_numeric = analizar_variables_numericas(
    df_clean, VARIABLES_NUMERICAS, TARGET_CONTINUO, TARGET_BINARIO, df_normality
)

print("\n📋 Resultados del análisis numérico:")
print(df_results_numeric[['Variable', 'Correlacion_Spearman', 'Fuerza_Spearman', 
                         'Significativa_Spearman', 'Cohens_d', 'Tamaño_Efecto']].to_string(index=False))

7. Análisis de Variables Categóricas

In [ ]:
print("\n" + "="*80)
print("FASE 7: ANÁLISIS DE VARIABLES CATEGÓRICAS")
print("="*80)

def analizar_variables_categoricas(df, variables, target_cont, target_bin, df_normality):
    """Análisis completo para variables categóricas"""
    resultados = []
    
    for i, var in enumerate(variables, 1):
        print(f"🔍 Analizando {i}/{len(variables)}: {var}")
        
        if var not in df.columns:
            print(f"  ⚠ Variable {var} no encontrada, saltando...")
            continue
            
        try:
            # Kruskal-Wallis
            grupos_kw = [grupo[target_cont].values for nombre, grupo in df.groupby(var) if len(grupo) > 5]
            
            if len(grupos_kw) > 1:
                kw_stat, kw_p = kruskal(*grupos_kw)
            else:
                kw_stat, kw_p = np.nan, np.nan
                
            # Chi-cuadrado
            tabla_contingencia = pd.crosstab(df[var], df[target_bin])
            if tabla_contingencia.shape[0] > 1 and tabla_contingencia.shape[1] > 1:
                chi2_stat, chi2_p, dof, esperado = chi2_contingency(tabla_contingencia)
            else:
                chi2_stat, chi2_p = np.nan, np.nan
            
            # Determinar test principal
            es_normal_target = df_normality[df_normality['Variable'] == target_cont]['Es_Normal'].values[0]
            test_principal = 'ANOVA' if es_normal_target else 'Kruskal-Wallis'
            p_value_principal = kw_p
            
            resultados.append({
                'Variable': var,
                'Test_Principal': test_principal,
                'p_value_Principal': round(p_value_principal, 6) if not np.isnan(p_value_principal) else np.nan,
                'Significativa_Principal': p_value_principal < 0.05 if not np.isnan(p_value_principal) else False,
                'p_value_Chi2': round(chi2_p, 6) if not np.isnan(chi2_p) else np.nan,
                'Significativa_Chi2': chi2_p < 0.05 if not np.isnan(chi2_p) else False,
                'n_Categorias': len(df[var].unique())
            })
            
        except Exception as e:
            print(f"  ✗ Error analizando {var}: {e}")
    
    return pd.DataFrame(resultados)

# Ejecutar análisis categórico
df_results_categorical = analizar_variables_categoricas(
    df_clean, VARIABLES_CATEGORICAS, TARGET_CONTINUO, TARGET_BINARIO, df_normality
)

print("\n📋 Resultados del análisis categórico:")
print(df_results_categorical[['Variable', 'Test_Principal', 'p_value_Principal', 
                            'Significativa_Principal', 'Significativa_Chi2']].to_string(index=False))

# ANÁLISIS CORREGIDO DE VARIABLES BINARIAS
print("\n" + "="*80)
print("ANÁLISIS CORREGIDO DE VARIABLES BINARIAS")
print("="*80)

# Ejecutar análisis corregido
print("🔍 Ejecutando análisis corregido de variables binarias...")
df_results_binarias_corregido = BinaryVariablesAnalyzerCorregido.analizar_binarias_corregido(
    df_clean, VARIABLES_BINARIAS, TARGET_CONTINUO, TARGET_BINARIO
)

# Mostrar resultados
if not df_results_binarias_corregido.empty:
    print("\n📊 RESULTADOS CORREGIDOS - VARIABLES BINARIAS:")
    print("="*60)
    columnas_mostrar = ['Variable', 'Test_Utilizado', 'p_valor', 'Significativa', 
                       'Diferencia_Medias', 'Cohens_d', 'PointBiserial_Corr']
    print(df_results_binarias_corregido[columnas_mostrar].to_string(index=False))
    
    # Interpretación
    BinaryVariablesAnalyzerCorregido.interpretar_resultados_binarios(df_results_binarias_corregido)
else:
    print("❌ No se pudieron generar resultados para variables binarias")

# COMPARACIÓN CON EL ANÁLISIS ANTERIOR (si existe)
if 'df_results_binarias' in locals() and not df_results_binarias.empty:
    print("\n" + "="*80)
    print("COMPARACIÓN: ANÁLISIS ANTERIOR vs CORREGIDO")
    print("="*80)
    
    print("ANÁLISIS ANTERIOR:")
    print(df_results_binarias[['Variable', 'Mann_Whitney_p', 'Significativa_MW', 'Cohens_d']].to_string(index=False))
    
    print("\nANÁLISIS CORREGIDO:")
    print(df_results_binarias_corregido[['Variable', 'Test_Utilizado', 'p_valor', 'Significativa', 'Cohens_d']].to_string(index=False))

7. 5: VERIFICACIÓN Y CORRECCIÓN DE MÉTRICAS BINARIAS

In [ ]:
print("\n" + "="*80)
print("FASE 7.5: VERIFICACIÓN Y CORRECCIÓN DE MÉTRICAS BINARIAS")
print("="*80)

# Asegurar que el directorio de salida existe ANTES de cualquier operación de guardado
import os
os.makedirs(config.OUTPUT_PATH, exist_ok=True)
print(f"📁 Directorio de salida verificado: {config.OUTPUT_PATH}")

class MetricasCorrector:
    """Clase para verificar y corregir cálculos de OR, RR y validar variables"""
    
    def __init__(self, output_path):
        self.output_path = output_path
        # Asegurar que el directorio existe
        os.makedirs(output_path, exist_ok=True)
    
    @staticmethod
    def verificar_codificacion_disciplinary_failure(df):
        """Verificación detallada de Disciplinary_failure"""
        print("\n🔍 VALIDACIÓN MANUAL DE DISCIPLINARY_FAILURE:")
        print("-" * 50)
        
        if 'Disciplinary_failure' not in df.columns:
            print("❌ Disciplinary_failure no encontrada en el dataset")
            return
        
        # Análisis descriptivo completo
        print("📊 DISTRIBUCIÓN DE DISCIPLINARY_FAILURE:")
        print(df['Disciplinary_failure'].value_counts())
        print(f"Proporciones: {df['Disciplinary_failure'].value_counts(normalize=True)}")
        
        print("\n📈 AUSENTISMO POR DISCIPLINARY_FAILURE:")
        ausentismo_por_grupo = df.groupby('Disciplinary_failure')['Absenteeism_hours'].agg([
            'count', 'mean', 'median', 'std', 'min', 'max'
        ]).round(2)
        print(ausentismo_por_grupo)
        
        # Tabla de contingencia detallada
        print("\n🎯 TABLA DE CONTINGENCIA COMPLETA:")
        tabla = pd.crosstab(df['Disciplinary_failure'], df['Ausentismo_Alto'], 
                           margins=True, margins_name="Total")
        print(tabla)
        
        # Cálculo manual de OR y RR
        print("\n🧮 CÁLCULO MANUAL DE OR Y RR:")
        if tabla.shape == (3, 3):  # Incluye totales
            a = tabla.iloc[0, 0]  # Grupo 0, Ausentismo Bajo
            b = tabla.iloc[0, 1]  # Grupo 0, Ausentismo Alto
            c = tabla.iloc[1, 0]  # Grupo 1, Ausentismo Bajo
            d = tabla.iloc[1, 1]  # Grupo 1, Ausentismo Alto
            
            # Odds Ratio manual
            odds_ratio_manual = (d / c) / (b / a) if (a > 0 and c > 0) else np.nan
            
            # Risk Ratio manual
            riesgo_grupo1 = d / (c + d) if (c + d) > 0 else 0
            riesgo_grupo0 = b / (a + b) if (a + b) > 0 else 0
            risk_ratio_manual = riesgo_grupo1 / riesgo_grupo0 if riesgo_grupo0 > 0 else np.nan
            
            print(f"  - Odds Ratio manual: {odds_ratio_manual:.4f}")
            print(f"  - Risk Ratio manual: {risk_ratio_manual:.4f}")
            print(f"  - Riesgo Grupo 0: {riesgo_grupo0:.4f}")
            print(f"  - Riesgo Grupo 1: {riesgo_grupo1:.4f}")
            
            # Interpretación
            if not np.isnan(odds_ratio_manual):
                if odds_ratio_manual > 1:
                    print("  📈 Interpretación OR: AUMENTA el riesgo de ausentismo alto")
                else:
                    print("  📉 Interpretación OR: DISMINUYE el riesgo de ausentismo alto")
    
    @staticmethod
    def corregir_calculos_or_rr(df, binary_vars, target_bin):
        """Recalcular correctamente OR y RR para todas las variables binarias"""
        print("\n🔧 CORRECCIÓN DE CÁLCULOS OR Y RR:")
        print("-" * 40)
        
        resultados_corregidos = []
        
        for var in binary_vars:
            if var not in df.columns:
                continue
                
            print(f"\n📊 Recalculando {var}:")
            
            try:
                # Tabla de contingencia sin totales
                tabla = pd.crosstab(df[var], df[target_bin])
                
                if tabla.shape != (2, 2):
                    print(f"  ⚠ Tabla no es 2x2: {tabla.shape}")
                    continue
                
                # Extraer valores asegurando el orden correcto
                # Grupo 0 (var=0), Grupo 1 (var=1)
                # Columna 0 (target=0), Columna 1 (target=1)
                a = tabla.iloc[0, 0]  # var=0, target=0
                b = tabla.iloc[0, 1]  # var=0, target=1  
                c = tabla.iloc[1, 0]  # var=1, target=0
                d = tabla.iloc[1, 1]  # var=1, target=1
                
                print(f"  - Tabla: [{a}, {b}; {c}, {d}]")
                
                # CÁLCULOS CORRECTOS:
                
                # 1. Risk Ratio (Ratio de Riesgos)
                riesgo_grupo1 = d / (c + d) if (c + d) > 0 else 0
                riesgo_grupo0 = b / (a + b) if (a + b) > 0 else 0
                risk_ratio = riesgo_grupo1 / riesgo_grupo0 if riesgo_grupo0 > 0 else np.nan
                
                # 2. Odds Ratio
                odds_grupo1 = d / c if c > 0 else np.nan
                odds_grupo0 = b / a if a > 0 else np.nan
                odds_ratio = odds_grupo1 / odds_grupo0 if not np.isnan(odds_grupo1) and not np.isnan(odds_grupo0) else np.nan
                
                # 3. Test exacto de Fisher
                try:
                    odds_ratio_fisher, fisher_p = fisher_exact(tabla)
                except:
                    odds_ratio_fisher, fisher_p = np.nan, np.nan
                
                print(f"  ✓ Risk Ratio corregido: {risk_ratio:.4f}")
                print(f"  ✓ Odds Ratio corregido: {odds_ratio:.4f}")
                print(f"  ✓ Odds Ratio Fisher: {odds_ratio_fisher:.4f}")
                print(f"  ✓ Riesgo Grupo 0: {riesgo_grupo0:.4f}")
                print(f"  ✓ Riesgo Grupo 1: {riesgo_grupo1:.4f}")
                
                resultados_corregidos.append({
                    'Variable': var,
                    'Risk_Ratio_Corregido': risk_ratio,
                    'Odds_Ratio_Corregido': odds_ratio,
                    'Odds_Ratio_Fisher': odds_ratio_fisher,
                    'Fisher_p': fisher_p,
                    'Riesgo_Grupo0': riesgo_grupo0,
                    'Riesgo_Grupo1': riesgo_grupo1,
                    'Diferencia_Riesgo': riesgo_grupo1 - riesgo_grupo0
                })
                
            except Exception as e:
                print(f"  ❌ Error calculando {var}: {e}")
        
        return pd.DataFrame(resultados_corregidos)
    
    def analizar_reason_absence_numeric(self, df):
        """Análisis detallado de Reason_absence_numeric"""
        print("\n🎯 ANÁLISIS DETALLADO DE REASON_ABSENCE_NUMERIC:")
        print("-" * 50)
        
        if 'Reason_absence_numeric' not in df.columns:
            print("❌ Reason_absence_numeric no encontrada")
            return
        
        # Distribución de valores
        print("📊 DISTRIBUCIÓN DE VALORES:")
        print(df['Reason_absence_numeric'].value_counts().sort_index())
        
        # Relación con ausentismo
        print("\n📈 AUSENTISMO POR RAZÓN:")
        ausentismo_por_razon = df.groupby('Reason_absence_numeric')['Absenteeism_hours'].agg([
            'count', 'mean', 'median', 'std', 'min', 'max'
        ]).round(2)
        print(ausentismo_por_razon)
        
        # Correlación detallada
        corr_spearman = df['Reason_absence_numeric'].corr(df['Absenteeism_hours'], method='spearman')
        corr_pearson = df['Reason_absence_numeric'].corr(df['Absenteeism_hours'], method='pearson')
        
        print(f"\n🔗 CORRELACIONES:")
        print(f"  - Spearman: {corr_spearman:.4f}")
        print(f"  - Pearson: {corr_pearson:.4f}")
        
        # Visualización de la relación
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=df, x='Reason_absence_numeric', y='Absenteeism_hours')
        plt.title('Distribución de Ausentismo por Reason_absence_numeric')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        # Asegurar que el directorio existe antes de guardar
        os.makedirs(self.output_path, exist_ok=True)
        plt.savefig(f'{self.output_path}/reason_absence_analysis.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print("✓ Gráfico de análisis guardado: reason_absence_analysis.png")

# Ejecutar verificaciones - PASAR output_path al constructor
corrector = MetricasCorrector(config.OUTPUT_PATH)

# 1. Validar Disciplinary_failure
corrector.verificar_codificacion_disciplinary_failure(df_clean)

# 2. Corregir cálculos OR y RR
df_or_rr_corregidos = corrector.corregir_calculos_or_rr(df_clean, VARIABLES_BINARIAS, TARGET_BINARIO)

# 3. Analizar Reason_absence_numeric
corrector.analizar_reason_absence_numeric(df_clean)

7. 6: ANÁLISIS DE DISTRIBUCIONES Y MULTICOLINEALIDAD

In [ ]:
print("\n" + "="*80)
print("FASE 7.6: ANÁLISIS DE DISTRIBUCIONES Y MULTICOLINEALIDAD")
print("="*80)

print("\n" + "="*80)
print("ANÁLISIS DE DISTRIBUCIONES Y MULTICOLINEALIDAD")
print("="*80)

class AnalisisDistribuciones:
    """Clase para analizar distribuciones y multicolinealidad"""
    
    @staticmethod
    def analizar_distribuciones_variables_significativas(df, variables):
        """Analizar distribuciones de variables significativas"""
        print("\n📊 ANÁLISIS DE DISTRIBUCIONES:")
        print("-" * 40)
        
        variables_analizar = [v for v in variables if v in df.columns]
        
        for var in variables_analizar:
            print(f"\n🔍 {var}:")
            data = df[var].dropna()
            
            # Estadísticas básicas
            print(f"  - n: {len(data)}")
            print(f"  - Media: {data.mean():.3f}")
            print(f"  - Mediana: {data.median():.3f}")
            print(f"  - Std: {data.std():.3f}")
            print(f"  - Mín: {data.min():.3f}")
            print(f"  - Máx: {data.max():.3f}")
            print(f"  - Asimetría: {stats.skew(data):.3f}")
            print(f"  - Curtosis: {stats.kurtosis(data):.3f}")
            
            # Test de normalidad
            if len(data) > 3:
                stat, p_valor = shapiro(data)
                print(f"  - Shapiro-Wilk: p = {p_valor:.6f} {'(Normal)' if p_valor > 0.05 else '(No normal)'}")
            
            # Sugerencia de transformación
            asimetria = abs(stats.skew(data))
            if asimetria > 1:
                print(f"  💡 Sugerencia: Transformación fuerte recomendada (asimetría: {asimetria:.3f})")
            elif asimetria > 0.5:
                print(f"  💡 Sugerencia: Transformación moderada considerada (asimetría: {asimetria:.3f})")
            else:
                print(f"  💡 Distribución relativamente simétrica (asimetría: {asimetria:.3f})")
    
    @staticmethod
    def aplicar_transformaciones(df, variables):
        """Aplicar y evaluar transformaciones"""
        print("\n🔄 EVALUACIÓN DE TRANSFORMACIONES:")
        print("-" * 40)
        
        resultados_transformaciones = []
        
        for var in variables:
            if var not in df.columns:
                continue
                
            data_original = df[var].dropna()
            if len(data_original) == 0:
                continue
            
            # Solo aplicar a variables continuas no binarias
            if data_original.nunique() > 2 and pd.api.types.is_numeric_dtype(data_original):
                print(f"\n📈 {var}:")
                
                # Transformaciones
                transformaciones = {
                    'Original': data_original,
                    'Log': np.log1p(data_original) if (data_original >= 0).all() else None,
                    'Raíz Cuadrada': np.sqrt(data_original) if (data_original >= 0).all() else None,
                    'Box-Cox': None
                }
                
                # Intentar Box-Cox
                try:
                    if (data_original > 0).all():
                        data_boxcox, _ = stats.boxcox(data_original)
                        transformaciones['Box-Cox'] = data_boxcox
                except:
                    pass
                
                for nombre, data_trans in transformaciones.items():
                    if data_trans is not None:
                        asimetria = stats.skew(data_trans)
                        resultados_transformaciones.append({
                            'Variable': var,
                            'Transformación': nombre,
                            'Asimetría': asimetria,
                            'Mejora': abs(asimetria) < abs(stats.skew(data_original))
                        })
                        print(f"  - {nombre}: asimetría = {asimetria:.3f}")
        
        return pd.DataFrame(resultados_transformaciones)
    
    @staticmethod
    def analizar_multicolinealidad(df, variables):
        """Analizar multicolinealidad entre variables predictoras"""
        print("\n🔗 ANÁLISIS DE MULTICOLINEALIDAD:")
        print("-" * 40)
        
        # Filtrar variables numéricas
        vars_numericas = [v for v in variables if v in df.columns and pd.api.types.is_numeric_dtype(df[v])]
        
        if len(vars_numericas) < 2:
            print("❌ No hay suficientes variables numéricas para analizar multicolinealidad")
            return
        
        # Matriz de correlación
        corr_matrix = df[vars_numericas].corr(method='spearman')
        
        # Encontrar correlaciones altas
        correlaciones_altas = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                corr_val = abs(corr_matrix.iloc[i, j])
                if corr_val > 0.7:
                    correlaciones_altas.append({
                        'Variable1': corr_matrix.columns[i],
                        'Variable2': corr_matrix.columns[j],
                        'Correlación': corr_val
                    })
        
        print("📊 CORRELACIONES ALTAS (> 0.7):")
        if correlaciones_altas:
            for corr in correlaciones_altas:
                print(f"  ⚠ {corr['Variable1']} - {corr['Variable2']}: {corr['Correlación']:.3f}")
        else:
            print("  ✅ No hay correlaciones altas problemáticas")
        
        # Heatmap de correlaciones
        plt.figure(figsize=(12, 10))
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
                   square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
        plt.title('MATRIZ DE CORRELACIÓN - MULTICOLINEALIDAD', fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(f'{config.OUTPUT_PATH}/multicolinealidad_analysis.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        print("✓ Heatmap de multicolinealidad guardado")
        
        return correlaciones_altas

# Variables significativas para análisis
VARIABLES_SIGNIFICATIVAS = [
    'Reason_absence_numeric', 'Disciplinary_failure', 'Social_drinker', 
    'Son', 'Transportation_expense'
]

# Ejecutar análisis
analizador = AnalisisDistribuciones()

# 1. Analizar distribuciones
analizador.analizar_distribuciones_variables_significativas(df_clean, VARIABLES_SIGNIFICATIVAS)

# 2. Evaluar transformaciones
df_transformaciones = analizador.aplicar_transformaciones(df_clean, VARIABLES_SIGNIFICATIVAS)

# 3. Analizar multicolinealidad
correlaciones_problematicas = analizador.analizar_multicolinealidad(df_clean, VARIABLES_SIGNIFICATIVAS)

7. 7: RESUMEN FINAL DE CORRECCIONES Y RECOMENDACIONES

In [ ]:
print("\n" + "="*80)
print("FASE 7.7: RESUMEN FINAL DE CORRECCIONES Y RECOMENDACIONES")
print("="*80)

print("\n" + "="*80)
print("RESUMEN FINAL DE CORRECCIONES Y RECOMENDACIONES")
print("="*80)

class ResumenCorrecciones:
    """Resumen final de todas las correcciones y recomendaciones"""
    
    @staticmethod
    def generar_resumen_completo(df_or_rr_corregidos, correlaciones_problematicas, df_transformaciones):
        """Generar resumen completo de correcciones"""
        
        print("\n🎯 RESUMEN EJECUTIVO DE CORRECCIONES:")
        print("="*50)
        
        # 1. Correcciones OR/RR
        print("\n1. ✅ CORRECCIONES OR/RR APLICADAS:")
        if not df_or_rr_corregidos.empty:
            for _, row in df_or_rr_corregidos.iterrows():
                print(f"   📊 {row['Variable']}:")
                print(f"      - Risk Ratio: {row['Risk_Ratio_Corregido']:.3f}")
                print(f"      - Odds Ratio: {row['Odds_Ratio_Corregido']:.3f}")
                
                # Interpretación corregida
                if row['Risk_Ratio_Corregido'] > 1:
                    print(f"      📈 AUMENTA el riesgo de ausentismo alto")
                else:
                    print(f"      📉 DISMINUYE el riesgo de ausentismo alto")
        
        # 2. Multicolinealidad
        print("\n2. 🔗 ANÁLISIS DE MULTICOLINEALIDAD:")
        if correlaciones_problematicas:
            print("   ⚠ CORRELACIONES PROBLEMÁTICAS DETECTADAS:")
            for corr in correlaciones_problematicas:
                print(f"      - {corr['Variable1']} ↔ {corr['Variable2']}: {corr['Correlación']:.3f}")
            print("   💡 RECOMENDACIÓN: Considerar eliminar una variable de cada par correlacionado")
        else:
            print("   ✅ No se detectaron problemas de multicolinealidad")
        
        # 3. Transformaciones
        print("\n3. 🔄 RECOMENDACIONES DE TRANSFORMACIONES:")
        if not df_transformaciones.empty:
            # Encontrar mejor transformación para cada variable
            variables_unicas = df_transformaciones['Variable'].unique()
            for var in variables_unicas:
                data_var = df_transformaciones[df_transformaciones['Variable'] == var]
                mejor_transform = data_var.loc[data_var['Asimetría'].abs().idxmin()]
                print(f"   📈 {var}:")
                print(f"      - Mejor transformación: {mejor_transform['Transformación']}")
                print(f"      - Asimetría resultante: {mejor_transform['Asimetría']:.3f}")
        
        # 4. Recomendaciones finales para modelo multivariable
        print("\n4. 🎯 RECOMENDACIONES FINALES PARA MODELO MULTIVARIABLE:")
        print("   🥇 VARIABLES PRIORITARIAS (alto impacto):")
        print("      - Reason_absence_numeric (Cohen's d = -0.630)")
        print("      - Disciplinary_failure (efecto significativo pero requiere validación)")
        
        print("\n   🥈 VARIABLES SECUNDARIAS (impacto moderado):")
        print("      - Social_drinker (Cohen's d = 0.374)")
        print("      - Son (Cohen's d = 0.226)")
        
        print("\n   🥉 VARIABLE A EVALUAR (impacto bajo):")
        print("      - Transportation_expense (Cohen's d = 0.032)")
        
        print("\n5. 🚨 ACCIONES CRÍTICAS ANTES DEL MODELO MULTIVARIABLE:")
        print("   ✅ Validar codificación de Disciplinary_failure")
        print("   ✅ Usar OR/RR corregidos en el análisis")
        print("   ✅ Considerar transformaciones para mejorar distribuciones")
        print("   ✅ Evaluar interacciones entre variables personales")
        print("   ✅ Realizar análisis de sensibilidad excluyendo Transportation_expense")

# Ejecutar resumen
resumen = ResumenCorrecciones()
resumen.generar_resumen_completo(df_or_rr_corregidos, correlaciones_problematicas, df_transformaciones)

print(f"\n📁 Todos los análisis detallados guardados en: {config.OUTPUT_PATH}")
print("✅ CORRECCIONES COMPLETADAS - Listo para análisis multivariable")

8. Visualizaciones (Modularizado)

In [ ]:
print("\n" + "="*80)
print("FASE 8: GENERACIÓN DE VISUALIZACIONES")
print("="*80)

class VisualizacionGenerator:
    def __init__(self, output_path):
        self.output_path = output_path
        import os
        os.makedirs(output_path, exist_ok=True)
        print(f"📁 Directorio de salida: {output_path}")
    
    def generar_heatmap_corregido(self, df, variables_personales, variables_laborales, target):
        """Heatmap corregido que solo usa variables numéricas verificadas"""
        print("📊 Generando heatmap de correlaciones CORREGIDO...")
        
        # Filtrar solo variables numéricas verificadas
        vars_numericas_personales = [var for var in variables_personales 
                                   if var in df.columns and pd.api.types.is_numeric_dtype(df[var])]
        vars_numericas_laborales = [var for var in variables_laborales 
                                  if var in df.columns and pd.api.types.is_numeric_dtype(df[var])]
        
        corr_vars = vars_numericas_personales + vars_numericas_laborales + [target]
        
        print(f"  - Variables personales numéricas: {len(vars_numericas_personales)}")
        print(f"  - Variables laborales numéricas: {len(vars_numericas_laborales)}")
        print(f"  - Total variables para heatmap: {len(corr_vars)}")
        
        if len(corr_vars) > 1:
            try:
                # Calcular matriz de correlación
                corr_matrix = df[corr_vars].corr(method='spearman')
                
                # Crear heatmap
                mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
                plt.figure(figsize=(16, 14))
                sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
                           square=True, fmt='.3f', cbar_kws={'shrink': 0.8})
                plt.title('MATRIZ DE CORRELACIÓN SPEARMAN - AUSENTISMO LABORAL\n(Variables Personales y Laborales Numéricas)', 
                         fontsize=16, fontweight='bold', pad=20)
                plt.tight_layout()
                plt.savefig(f'{self.output_path}/1_heatmap_correlaciones.png', dpi=300, bbox_inches='tight')
                plt.close()
                print("✓ Heatmap de correlaciones guardado")
                
            except Exception as e:
                print(f"✗ Error generando heatmap: {e}")
                self._generar_heatmap_simplificado(df, corr_vars)
        else:
            print("⚠ No hay suficientes variables numéricas para generar heatmap")
    
    def _generar_heatmap_simplificado(self, df, corr_vars):
        """Heatmap simplificado de respaldo"""
        try:
            print("  🔧 Generando heatmap simplificado...")
            corr_matrix = df[corr_vars].corr(method='spearman')
            plt.figure(figsize=(12, 10))
            sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
            plt.title('Correlación Spearman - Versión Simplificada', fontsize=14)
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/1_heatmap_simplificado.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Heatmap simplificado guardado")
        except Exception as e:
            print(f"✗ Error en heatmap simplificado: {e}")
    
    def generar_importancia_variables(self, df_results):
        print("📊 Generando gráfico de importancia...")
        if df_results.empty:
            print("⚠ No hay datos para generar gráfico de importancia")
            return
        
        try:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
            
            # Gráfico de correlación
            corr_data = df_results.sort_values('abs_corr', ascending=True)
            colors = ['red' if sig else 'gray' for sig in corr_data['Significativa_Spearman']]
            ax1.barh(corr_data['Variable'], corr_data['abs_corr'], color=colors, alpha=0.7)
            ax1.set_xlabel('Correlación Absoluta (Spearman)')
            ax1.set_title('IMPORTANCIA POR CORRELACIÓN', fontweight='bold')
            ax1.grid(axis='x', alpha=0.3)
            
            # Gráfico de tamaño del efecto
            effect_data = df_results.sort_values('abs_effect', ascending=True)
            colors_effect = ['red' if sig else 'gray' for sig in effect_data['Significativa_Medias']]
            ax2.barh(effect_data['Variable'], effect_data['abs_effect'], color=colors_effect, alpha=0.7)
            ax2.set_xlabel('Tamaño del Efecto Absoluto (Cohen d)')
            ax2.set_title('IMPORTANCIA POR TAMAÑO DEL EFECTO', fontweight='bold')
            ax2.grid(axis='x', alpha=0.3)
            
            plt.suptitle('COMPARACIÓN DE IMPORTANCIA DE VARIABLES', fontsize=16, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/2_importancia_variables.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Gráfico de importancia de variables guardado")
        except Exception as e:
            print(f"✗ Error generando gráfico de importancia: {e}")
    
    def generar_scatter_variables_significativas(self, df, df_results, target, max_variables=6):
        print("📊 Generando scatter plots de variables significativas...")
        if df_results.empty:
            print("⚠ No hay resultados para generar scatter plots")
            return
        
        try:
            # Filtrar solo variables significativas
            sig_vars_df = df_results[df_results['Significativa_Spearman']]
            if sig_vars_df.empty:
                print("⚠ No hay variables significativas para scatter plots")
                return
            
            sig_vars = sig_vars_df.nlargest(max_variables, 'abs_corr')['Variable'].tolist()
            
            # Filtrar solo variables numéricas
            sig_vars_numericas = [var for var in sig_vars 
                                if var in df.columns and pd.api.types.is_numeric_dtype(df[var])]
            
            n_vars = len(sig_vars_numericas)
            if n_vars == 0:
                print("⚠ No hay variables numéricas significativas para scatter plots")
                return
            
            n_cols = min(3, n_vars)
            n_rows = (n_vars + n_cols - 1) // n_cols
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
            
            if n_rows == 1 and n_cols == 1:
                axes = [axes]
            elif n_rows == 1:
                axes = axes
            else:
                axes = axes.flatten()
            
            for i, var in enumerate(sig_vars_numericas):
                if i < len(axes):
                    sns.regplot(data=df, x=var, y=target, ax=axes[i], 
                               scatter_kws={'alpha':0.6, 's':30}, 
                               line_kws={'color':'red'}, ci=95)
                    stats_row = df_results[df_results['Variable'] == var].iloc[0]
                    corr = stats_row['Correlacion_Spearman']
                    fuerza = stats_row['Fuerza_Spearman']
                    direccion = "Positiva" if corr > 0 else "Negativa"
                    title_text = f'{var}\nρ = {corr:.3f} ({fuerza}, {direccion})'
                    axes[i].set_title(title_text, fontweight='bold', fontsize=10)
                    axes[i].set_xlabel(var)
                    axes[i].set_ylabel('Horas de Ausentismo')
            
            for j in range(len(sig_vars_numericas), len(axes)):
                axes[j].set_visible(False)
            
            plt.suptitle('RELACIÓN DE VARIABLES MÁS SIGNIFICATIVAS CON AUSENTISMO', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/3_scatter_variables_significativas.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Scatter plots de variables significativas guardados")
        except Exception as e:
            print(f"✗ Error generando scatter plots: {e}")
    
    def generar_boxplots_categoricas(self, df, df_results, target):
        print("📊 Generando boxplots de variables categóricas...")
        if df_results.empty:
            print("⚠ No hay resultados categóricos para generar boxplots")
            return
        
        try:
            # Filtrar solo variables categóricas significativas
            sig_cat_vars = df_results[df_results['Significativa_Principal']]['Variable'].tolist()
            if not sig_cat_vars:
                print("⚠ No hay variables categóricas significativas")
                return
            
            n_cat_vars = len(sig_cat_vars)
            n_cols = min(2, n_cat_vars)
            n_rows = (n_cat_vars + n_cols - 1) // n_cols
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
            
            if n_rows == 1 and n_cols == 1:
                axes = [axes]
            elif n_rows == 1:
                axes = axes
            else:
                axes = axes.flatten()
            
            for i, cat_var in enumerate(sig_cat_vars):
                if i < len(axes) and cat_var in df.columns:
                    order = df.groupby(cat_var)[target].median().sort_values(ascending=False).index
                    sns.boxplot(data=df, x=cat_var, y=target, ax=axes[i], order=order)
                    axes[i].set_title(f'Ausentismo por {cat_var}', fontweight='bold')
                    axes[i].tick_params(axis='x', rotation=45)
                    axes[i].set_xlabel(cat_var)
                    axes[i].set_ylabel('Horas de Ausentismo')
            
            for j in range(len(sig_cat_vars), len(axes)):
                axes[j].set_visible(False)
            
            plt.suptitle('DISTRIBUCIÓN DE AUSENTISMO POR VARIABLES CATEGÓRICAS', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/4_boxplots_categoricas.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Boxplots de variables categóricas guardados")
        except Exception as e:
            print(f"✗ Error generando boxplots: {e}")
    
    def generar_grafico_correlaciones_impacto(self, df_results, variables_personales, variables_laborales):
        """Gráfico 5 corregido: muestra solo variables personales y laborales significativas"""
        print("📊 Generando gráfico de correlaciones de impacto...")
        if df_results.empty:
            print("⚠ No hay resultados para generar gráfico de impacto")
            return
        
        try:
            # FILTRAR SOLO VARIABLES SIGNIFICATIVAS que son personales o laborales
            df_significativas = df_results[
                df_results['Significativa_Spearman'] & 
                (df_results['Variable'].isin(variables_personales + variables_laborales))
            ].copy()
            
            if df_significativas.empty:
                print("⚠ No hay variables personales o laborales significativas para el gráfico de impacto")
                return
                
            # Ordenar por correlación absoluta
            df_plot = df_significativas.sort_values('abs_corr', ascending=True)
            
            plt.figure(figsize=(14, 10))
            
            # Asignar colores según tipo de variable
            colors = []
            for _, row in df_plot.iterrows():
                if row['Variable'] in variables_personales:
                    colors.append((0.2, 0.4, 0.8, 0.8))  # Azul para personales
                else:
                    colors.append((0.8, 0.4, 0.2, 0.8))  # Naranja para laborales
            
            # Crear barras
            bars = plt.barh(df_plot['Variable'], df_plot['Correlacion_Spearman'], color=colors, alpha=0.8)
            
            # Añadir valores en las barras
            for i, (value, variable) in enumerate(zip(df_plot['Correlacion_Spearman'], df_plot['Variable'])):
                if value >= 0:
                    plt.text(value + 0.01, i, f'{value:.3f}', va='center', ha='left', fontweight='bold', fontsize=10)
                else:
                    plt.text(value - 0.01, i, f'{value:.3f}', va='center', ha='right', fontweight='bold', fontsize=10)
            
            plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
            plt.xlabel('Coeficiente de Correlación (Spearman)', fontsize=12, fontweight='bold')
            
            # Título indicando que solo muestra variables personales y laborales significativas
            plt.title('IMPACTO DE VARIABLES PERSONALES Y LABORALES EN EL AUSENTISMO\n(Solo variables significativas p < 0.05)', 
                     fontsize=16, fontweight='bold', pad=20)
            
            # Añadir anotaciones de efecto y tipo
            for i, (_, row) in enumerate(df_plot.iterrows()):
                effect_text = "Aumenta ausentismo" if row['Correlacion_Spearman'] > 0 else "Disminuye ausentismo"
                tipo = "PERSONAL" if row['Variable'] in variables_personales else "LABORAL"
                color = 'darkblue' if tipo == 'PERSONAL' else 'darkorange'
                plt.annotate(f"{effect_text} ({tipo})", 
                            xy=(row['Correlacion_Spearman'], i), 
                            xytext=(5, 0), 
                            textcoords='offset points', 
                            ha='left', va='center', 
                            fontsize=9, color=color, fontweight='bold')
            
            # Añadir leyenda
            from matplotlib.patches import Patch
            legend_elements = [
                Patch(facecolor=(0.2, 0.4, 0.8, 0.8), label='Variables Personales'),
                Patch(facecolor=(0.8, 0.4, 0.2, 0.8), label='Variables Laborales')
            ]
            plt.legend(handles=legend_elements, loc='lower right')
            
            plt.grid(axis='x', alpha=0.3)
            plt.tight_layout()
            plt.savefig(f'{self.output_path}/5_grafico_correlaciones_impacto.png', dpi=300, bbox_inches='tight')
            plt.close()
            print("✓ Gráfico de correlaciones de impacto guardado")
        except Exception as e:
            print(f"✗ Error generando gráfico de impacto: {e}")
    
    def generar_todas_visualizaciones(self, df, df_results_numeric, df_results_categorical, target, variables_personales, variables_laborales):
        print("\n🎨 GENERANDO TODAS LAS VISUALIZACIONES...")
        print("-" * 50)
        
        # Heatmap corregido
        self.generar_heatmap_corregido(df, variables_personales, variables_laborales, target)
        
        # Gráficos que usan resultados estadísticos
        self.generar_importancia_variables(df_results_numeric)
        self.generar_scatter_variables_significativas(df, df_results_numeric, target)
        self.generar_boxplots_categoricas(df, df_results_categorical, target)
        self.generar_grafico_correlaciones_impacto(df_results_numeric, variables_personales, variables_laborales)
        
        print("\n✅ TODAS LAS VISUALIZACIONES GENERADAS EXITOSAMENTE!")

# EJECUCIÓN CORREGIDA
try:
    # Asegurar que tenemos la columna Fuerza_Num
    if 'Fuerza_Num' not in df_results_numeric.columns:
        def clasificar_correlacion(corr):
            abs_corr = abs(corr)
            if abs_corr < 0.1: return "Muy débil", 1
            elif abs_corr < 0.3: return "Débil", 2
            elif abs_corr < 0.5: return "Moderada", 3
            elif abs_corr < 0.7: return "Fuerte", 4
            else: return "Muy fuerte", 5
        
        resultados = df_results_numeric['Correlacion_Spearman'].apply(clasificar_correlacion)
        df_temp = pd.DataFrame(resultados.tolist(), columns=['Fuerza_Spearman', 'Fuerza_Num'], index=df_results_numeric.index)
        df_results_numeric = pd.concat([df_results_numeric, df_temp], axis=1)

    print(f"📁 Creando visualizaciones en: {config.OUTPUT_PATH}")
    viz = VisualizacionGenerator(config.OUTPUT_PATH)
    
    # Llamada CORREGIDA - solo pasamos las listas necesarias
    viz.generar_todas_visualizaciones(
        df_clean, 
        df_results_numeric, 
        df_results_categorical, 
        TARGET_CONTINUO,
        VARIABLES_PERSONALES,
        VARIABLES_LABORALES
    )

    print(f"\n📁 Visualizaciones guardadas en: {config.OUTPUT_PATH}")
    print("✓ 1_heatmap_correlaciones.png")
    print("✓ 2_importancia_variables.png") 
    print("✓ 3_scatter_variables_significativas.png")
    print("✓ 4_boxplots_categoricas.png")
    print("✓ 5_grafico_correlaciones_impacto.png")

except Exception as e:
    print(f"❌ Error en visualizaciones: {e}")
    print("Generando visualización mínima de respaldo...")
    try:
        # Respaldo seguro - solo variables numéricas verificadas
        vars_numericas_seguras = [var for var in VARIABLES_PERSONALES + VARIABLES_LABORALES 
                                if var in df_clean.columns and pd.api.types.is_numeric_dtype(df_clean[var])]
        vars_para_respaldo = vars_numericas_seguras[:8] + [TARGET_CONTINUO]
        
        if len(vars_para_respaldo) > 2:
            plt.figure(figsize=(10, 8))
            corr_matrix = df_clean[vars_para_respaldo].corr(method='spearman')
            sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r')
            plt.title('Correlación con Ausentismo - Respaldo')
            plt.tight_layout()
            plt.savefig(f'{config.OUTPUT_PATH}/backup_heatmap.png')
            plt.close()
            print("✓ Visualización de respaldo guardada")
    except Exception as backup_error:
        print(f"✗ Error en visualización de respaldo: {backup_error}")

In [ ]:
print("\n" + "="*80)
print("INFORME EJECUTIVO COMPLETO - VARIABLES PERSONALES Y LABORALES")
print("="*80)

class InformeEjecutivo:
    """Clase para generar informe ejecutivo completo clasificado por tipo de variable"""
    
    @staticmethod
    def generar_informe_completo(df_results_numeric, df_results_categorical, df_results_binarias_corregido, 
                                variables_personales, variables_laborales, variables_temporales):
        """Generar informe ejecutivo completo clasificado por tipo de variable"""
        
        print("\n📋 INFORME EJECUTIVO COMPLETO")
        print("="*60)
        
        # 1. VARIABLES PERSONALES SIGNIFICATIVAS
        print("\n🎯 VARIABLES PERSONALES SIGNIFICATIVAS:")
        print("-" * 40)
        vars_personales_significativas = []
        for var in variables_personales:
            if var in df_results_numeric['Variable'].values:
                resultado = df_results_numeric[df_results_numeric['Variable'] == var].iloc[0]
                if resultado['Significativa_Spearman']:
                    # CORREGIDO: Extraer correctamente la fuerza
                    fuerza_texto = resultado['Fuerza_Spearman']
                    if not isinstance(fuerza_texto, str):
                        # Si es una serie, extraer el valor
                        fuerza_texto = str(fuerza_texto).split()[-1] if len(str(fuerza_texto).split()) > 1 else "Débil"
                    
                    direccion = "AUMENTA" if resultado['Correlacion_Spearman'] > 0 else "DISMINUYE"
                    
                    vars_personales_significativas.append({
                        'Variable': var,
                        'Correlacion': resultado['Correlacion_Spearman'],
                        'Direccion': direccion,
                        'Fuerza': fuerza_texto,
                        'p_valor': resultado['p_value_Spearman'],
                        'Cohens_d': resultado['Cohens_d'],
                        'Tamaño_Efecto_Cohens': resultado['Tamaño_Efecto']
                    })
                    print(f"✓ {var}: {direccion} ausentismo")
                    print(f"  - ρ = {resultado['Correlacion_Spearman']:.3f} ({fuerza_texto}), p = {resultado['p_value_Spearman']:.6f}")
                    print(f"  - Cohen's d = {resultado['Cohens_d']:.3f} ({resultado['Tamaño_Efecto']})")
        
        if not vars_personales_significativas:
            print("  No hay variables personales significativas")
        
        # 2. VARIABLES LABORALES SIGNIFICATIVAS
        print("\n💼 VARIABLES LABORALES SIGNIFICATIVAS:")
        print("-" * 40)
        vars_laborales_significativas = []
        for var in variables_laborales:
            if var in df_results_numeric['Variable'].values:
                resultado = df_results_numeric[df_results_numeric['Variable'] == var].iloc[0]
                if resultado['Significativa_Spearman']:
                    # CORREGIDO: Extraer correctamente la fuerza
                    fuerza_texto = resultado['Fuerza_Spearman']
                    if not isinstance(fuerza_texto, str):
                        # Si es una serie, extraer el valor
                        fuerza_texto = str(fuerza_texto).split()[-1] if len(str(fuerza_texto).split()) > 1 else "Débil"
                    
                    direccion = "AUMENTA" if resultado['Correlacion_Spearman'] > 0 else "DISMINUYE"
                    
                    vars_laborales_significativas.append({
                        'Variable': var,
                        'Correlacion': resultado['Correlacion_Spearman'],
                        'Direccion': direccion,
                        'Fuerza': fuerza_texto,
                        'p_valor': resultado['p_value_Spearman'],
                        'Cohens_d': resultado['Cohens_d'],
                        'Tamaño_Efecto_Cohens': resultado['Tamaño_Efecto']
                    })
                    print(f"✓ {var}: {direccion} ausentismo")
                    print(f"  - ρ = {resultado['Correlacion_Spearman']:.3f} ({fuerza_texto}), p = {resultado['p_value_Spearman']:.6f}")
                    print(f"  - Cohen's d = {resultado['Cohens_d']:.3f} ({resultado['Tamaño_Efecto']})")
        
        if not vars_laborales_significativas:
            print("  No hay variables laborales significativas")
        
        # 3. VARIABLES BINARIAS SIGNIFICATIVAS
        print("\n🔘 VARIABLES BINARIAS SIGNIFICATIVAS:")
        print("-" * 40)
        if df_results_binarias_corregido is not None and not df_results_binarias_corregido.empty:
            for _, row in df_results_binarias_corregido[df_results_binarias_corregido['Significativa']].iterrows():
                # Clasificar si es personal o laboral
                tipo = "PERSONAL" if row['Variable'] in ['Social_drinker', 'Social_smoker'] else "LABORAL"
                print(f"✓ {row['Variable']} ({tipo}): {row['Interpretacion_Direccion']} ausentismo")
                print(f"   - Test: {row['Test_Utilizado']}, p = {row['p_valor']:.6f}")
                print(f"   - Diferencia: {row['Media_Grupo1']:.2f} vs {row['Media_Grupo0']:.2f} horas")
                print(f"   - Cohen's d = {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
                if not np.isnan(row['PointBiserial_Corr']):
                    print(f"   - Correlación Punto-Biserial: {row['PointBiserial_Corr']:.3f} ({row['Fuerza_PointBiserial']})")
                if not np.isnan(row['Odds_Ratio']):
                    print(f"   - Odds Ratio: {row['Odds_Ratio']:.3f}")
        
        # 4. RESUMEN PARA REGRESIONES MULTIVARIABLES
        print("\n📈 VARIABLES CANDIDATAS PARA REGRESIONES MULTIVARIABLES:")
        print("-" * 50)
        
        todas_significativas = vars_personales_significativas + vars_laborales_significativas
        
        if todas_significativas:
            # Ordenar por fuerza de correlación absoluta
            todas_significativas.sort(key=lambda x: abs(x['Correlacion']), reverse=True)
            
            print("Variables recomendadas para modelo multivariable (ordenadas por importancia):")
            for i, var in enumerate(todas_significativas[:10], 1):
                print(f"{i}. {var['Variable']}:")
                print(f"   - ρ = {var['Correlacion']:.3f} ({var['Fuerza']}, {var['Direccion']})")
                print(f"   - Cohen's d = {var['Cohens_d']:.3f} ({var['Tamaño_Efecto_Cohens']})")
                print(f"   - p = {var['p_valor']:.6f}")
            
            print(f"\n📊 RESUMEN ESTADÍSTICO:")
            print(f"   - Total variables significativas: {len(todas_significativas)}")
            print(f"   - Variables personales: {len(vars_personales_significativas)}")
            print(f"   - Variables laborales: {len(vars_laborales_significativas)}")
            
            # Estadísticas de efecto
            cohens_d_values = [var['Cohens_d'] for var in todas_significativas]
            print(f"   - Cohen's d promedio: {np.mean(cohens_d_values):.3f}")
            print(f"   - Cohen's d máximo: {max(cohens_d_values):.3f}")
            print(f"   - Cohen's d mínimo: {min(cohens_d_values):.3f}")
        else:
            print("No hay variables significativas para recomendar")
        
        # 5. GUARDAR INFORME DETALLADO
        InformeEjecutivo.guardar_informe_detallado(vars_personales_significativas, vars_laborales_significativas, 
                                                  df_results_binarias_corregido, config.OUTPUT_PATH)
    
    @staticmethod
    def guardar_informe_detallado(vars_personales, vars_laborales, df_binarias, output_path):
        """Guardar informe detallado en archivo"""
        
        with open(f'{output_path}/INFORME_EJECUTIVO_DETALLADO.txt', 'w', encoding='utf-8') as f:
            f.write("INFORME EJECUTIVO DETALLADO - ANÁLISIS DE AUSENTISMO LABORAL\n")
            f.write("="*70 + "\n\n")
            
            f.write("RESUMEN EJECUTIVO:\n")
            f.write(f"- Variables personales significativas: {len(vars_personales)}\n")
            f.write(f"- Variables laborales significativas: {len(vars_laborales)}\n")
            if df_binarias is not None:
                binarias_significativas = len(df_binarias[df_binarias['Significativa']]) if not df_binarias.empty else 0
                f.write(f"- Variables binarias significativas: {binarias_significativas}\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("VARIABLES PERSONALES SIGNIFICATIVAS:\n")
            f.write("="*70 + "\n")
            for var in vars_personales:
                f.write(f"✓ {var['Variable']}: {var['Direccion']} el ausentismo\n")
                f.write(f"  - Correlación (Spearman): {var['Correlacion']:.3f} ({var['Fuerza']})\n")
                f.write(f"  - p-valor: {var['p_valor']:.6f}\n")
                f.write(f"  - Cohen's d: {var['Cohens_d']:.3f} ({var['Tamaño_Efecto_Cohens']})\n")
                f.write(f"  - Interpretación: {'Mayor valor de esta variable se asocia con ' + var['Direccion'].lower() + ' ausentismo'}\n\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("VARIABLES LABORALES SIGNIFICATIVAS:\n")
            f.write("="*70 + "\n")
            for var in vars_laborales:
                f.write(f"✓ {var['Variable']}: {var['Direccion']} el ausentismo\n")
                f.write(f"  - Correlación (Spearman): {var['Correlacion']:.3f} ({var['Fuerza']})\n")
                f.write(f"  - p-valor: {var['p_valor']:.6f}\n")
                f.write(f"  - Cohen's d: {var['Cohens_d']:.3f} ({var['Tamaño_Efecto_Cohens']})\n")
                f.write(f"  - Interpretación: {'Mayor valor de esta variable se asocia con ' + var['Direccion'].lower() + ' ausentismo'}\n\n")
            
            if df_binarias is not None and not df_binarias.empty:
                f.write("\n" + "="*70 + "\n")
                f.write("VARIABLES BINARIAS SIGNIFICATIVAS:\n")
                f.write("="*70 + "\n")
                for _, row in df_binarias[df_binarias['Significativa']].iterrows():
                    f.write(f"✓ {row['Variable']}: {row['Interpretacion_Direccion']} el ausentismo\n")
                    f.write(f"  - Test utilizado: {row['Test_Utilizado']}\n")
                    f.write(f"  - p-valor: {row['p_valor']:.6f}\n")
                    f.write(f"  - Diferencia de medias: {row['Media_Grupo1']:.2f} vs {row['Media_Grupo0']:.2f} horas\n")
                    f.write(f"  - Tamaño del efecto (Cohen's d): {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})\n")
                    if not np.isnan(row['PointBiserial_Corr']):
                        f.write(f"  - Correlación Punto-Biserial: {row['PointBiserial_Corr']:.3f} ({row['Fuerza_PointBiserial']})\n")
                    if not np.isnan(row['Odds_Ratio']):
                        f.write(f"  - Odds Ratio: {row['Odds_Ratio']:.3f}\n")
                    if not np.isnan(row['Risk_Ratio']):
                        f.write(f"  - Risk Ratio: {row['Risk_Ratio']:.3f}\n\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("RECOMENDACIONES PARA ANÁLISIS MULTIVARIABLE:\n")
            f.write("="*70 + "\n")
            f.write("Variables recomendadas para regresiones lineales multivariables:\n")
            todas_vars = vars_personales + vars_laborales
            todas_vars.sort(key=lambda x: abs(x['Correlacion']), reverse=True)
            
            for i, var in enumerate(todas_vars[:10], 1):
                f.write(f"{i}. {var['Variable']}\n")
                f.write(f"   - Correlación: ρ = {var['Correlacion']:.3f} ({var['Direccion']})\n")
                f.write(f"   - Tamaño efecto: Cohen's d = {var['Cohens_d']:.3f} ({var['Tamaño_Efecto_Cohens']})\n")
                f.write(f"   - Significancia: p = {var['p_valor']:.6f}\n")
            
            f.write("\nConsideraciones para el modelo multivariable:\n")
            f.write("- Evaluar multicolinealidad entre variables predictoras\n")
            f.write("- Considerar interacciones entre variables personales y laborales\n")
            f.write("- Validar supuestos de normalidad y homocedasticidad\n")
            f.write("- Considerar transformaciones si es necesario\n")
            f.write("- Priorizar variables con mayor Cohen's d para mayor poder explicativo\n")

# Ejecutar informe ejecutivo
InformeEjecutivo.generar_informe_completo(
    df_results_numeric,
    df_results_categorical, 
    df_results_binarias_corregido,
    VARIABLES_PERSONALES,
    VARIABLES_LABORALES, 
    VARIABLES_TEMPORALES
)

print(f"\n📄 Informe ejecutivo detallado guardado en: {config.OUTPUT_PATH}/INFORME_EJECUTIVO_DETALLADO.txt")

9. Exportación de Resultados

In [ ]:
print("\n" + "="*80)
print("FASE 9: EXPORTACIÓN DE RESULTADOS (ACTUALIZADA)")
print("="*80)

class ExportadorResultados:
    """Clase para exportar resultados de manera organizada - VERSIÓN ACTUALIZADA"""
    
    def __init__(self, output_path):
        self.output_path = output_path
    
    def exportar_resultados_completos(self, df_numeric, df_categorical, df_normality, df_clean, 
                                    df_binarias_corregido, df_or_rr_corregidos=None, df_transformaciones=None,
                                    correlaciones_problematicas=None):
        """Exportar todos los resultados incluyendo las correcciones"""
        print("💾 Exportando resultados completos (INCLUYENDO CORRECCIONES)...")
        
        try:
            # Crear directorio si no existe
            os.makedirs(self.output_path, exist_ok=True)
            
            # Exportar DataFrames individuales ORIGINALES
            print("  📁 Exportando resultados originales...")
            df_numeric.to_csv(f'{self.output_path}/resultados_numericos.csv', index=False, encoding='utf-8-sig')
            df_categorical.to_csv(f'{self.output_path}/resultados_categoricos.csv', index=False, encoding='utf-8-sig')
            df_normality.to_csv(f'{self.output_path}/analisis_normalidad.csv', index=False)
            
            if df_binarias_corregido is not None:
                df_binarias_corregido.to_csv(f'{self.output_path}/analisis_binarias_corregido.csv', index=False, encoding='utf-8-sig')
            
            df_clean.to_parquet(f'{self.output_path}/dataset_analisis.parquet', index=False)
            
            # Exportar DataFrames NUEVOS de correcciones
            print("  📁 Exportando correcciones y análisis adicionales...")
            if df_or_rr_corregidos is not None and not df_or_rr_corregidos.empty:
                df_or_rr_corregidos.to_csv(f'{self.output_path}/correcciones_or_rr.csv', index=False, encoding='utf-8-sig')
                print("    ✓ correcciones_or_rr.csv")
            
            if df_transformaciones is not None and not df_transformaciones.empty:
                df_transformaciones.to_csv(f'{self.output_path}/analisis_transformaciones.csv', index=False, encoding='utf-8-sig')
                print("    ✓ analisis_transformaciones.csv")
            
            # Exportar análisis de multicolinealidad si está disponible
            if correlaciones_problematicas is not None:
                with open(f'{self.output_path}/multicolinealidad_alertas.txt', 'w', encoding='utf-8') as f:
                    f.write("ALERTAS DE MULTICOLINEALIDAD\n")
                    f.write("="*40 + "\n\n")
                    if correlaciones_problematicas:
                        f.write("CORRELACIONES PROBLEMÁTICAS DETECTADAS:\n")
                        for corr in correlaciones_problematicas:
                            f.write(f"- {corr['Variable1']} ↔ {corr['Variable2']}: {corr['Correlación']:.3f}\n")
                    else:
                        f.write("No se detectaron correlaciones problemáticas (> 0.7)\n")
                print("    ✓ multicolinealidad_alertas.txt")
            
            # Crear reporte consolidado ACTUALIZADO
            self._crear_reporte_consolidado(df_numeric, df_categorical, df_binarias_corregido, 
                                          df_or_rr_corregidos, df_transformaciones, correlaciones_problematicas)
            
            # Crear archivo de metadatos del análisis
            self._crear_metadatos_analisis()
            
            print("✓ Todos los archivos exportados correctamente")
            
        except Exception as e:
            print(f"✗ Error en exportación: {e}")
    
    def _crear_reporte_consolidado(self, df_numeric, df_categorical, df_binarias_corregido, 
                                 df_or_rr_corregidos, df_transformaciones, correlaciones_problematicas):
        """Crear reporte ejecutivo consolidado ACTUALIZADO"""
        print("📋 Generando reporte ejecutivo actualizado...")
        
        # Variables significativas
        vars_significativas = df_numeric[df_numeric['Significativa_Spearman']]
        vars_positivas = vars_significativas[vars_significativas['Correlacion_Spearman'] > 0]
        vars_negativas = vars_significativas[vars_significativas['Correlacion_Spearman'] < 0]
        
        with open(f'{self.output_path}/REPORTE_EJECUTIVO_ACTUALIZADO.txt', 'w', encoding='utf-8') as f:
            f.write("REPORTE EJECUTIVO ACTUALIZADO - ANÁLISIS DE AUSENTISMO LABORAL\n")
            f.write("="*70 + "\n\n")
            
            f.write("RESUMEN EJECUTIVO CON CORRECCIONES APLICADAS:\n")
            f.write(f"- Variables personales significativas: {len(vars_positivas[vars_positivas['Variable'].isin(VARIABLES_PERSONALES)])}\n")
            f.write(f"- Variables laborales significativas: {len(vars_negativas[vars_negativas['Variable'].isin(VARIABLES_LABORALES)])}\n")
            if df_binarias_corregido is not None:
                binarias_significativas = len(df_binarias_corregido[df_binarias_corregido['Significativa']]) if not df_binarias_corregido.empty else 0
                f.write(f"- Variables binarias significativas: {binarias_significativas}\n")
            
            # Añadir información de correcciones
            if df_or_rr_corregidos is not None and not df_or_rr_corregidos.empty:
                f.write(f"- Correcciones OR/RR aplicadas: {len(df_or_rr_corregidos)} variables\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("VARIABLES NUMÉRICAS SIGNIFICATIVAS QUE AUMENTAN AUSENTISMO:\n")
            for _, row in vars_positivas.iterrows():
                f.write(f"✓ {row['Variable']}: ρ = {row['Correlacion_Spearman']:.3f} ({row['Fuerza_Spearman']})\n")
                f.write(f"  - Cohen's d: {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})\n")
                f.write(f"  - p-valor: {row['p_value_Spearman']:.6f}\n")
            
            f.write("\nVARIABLES NUMÉRICAS SIGNIFICATIVAS QUE DISMINUYEN AUSENTISMO:\n")
            for _, row in vars_negativas.iterrows():
                f.write(f"✓ {row['Variable']}: ρ = {row['Correlacion_Spearman']:.3f} ({row['Fuerza_Spearman']})\n")
                f.write(f"  - Cohen's d: {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})\n")
                f.write(f"  - p-valor: {row['p_value_Spearman']:.6f}\n")
            
            # Añadir interpretación de variables binarias CORREGIDAS
            if df_binarias_corregido is not None and not df_binarias_corregido.empty:
                f.write("\nVARIABLES BINARIAS SIGNIFICATIVAS (ANÁLISIS CORREGIDO):\n")
                for _, row in df_binarias_corregido[df_binarias_corregido['Significativa']].iterrows():
                    efecto = row['Interpretacion_Direccion']
                    f.write(f"✓ {row['Variable']}: {efecto} el ausentismo (p={row['p_valor']:.4f}, Test: {row['Test_Utilizado']})\n")
                    f.write(f"  - Cohen's d: {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})\n")
                    if not np.isnan(row['Odds_Ratio']):
                        f.write(f"  - Odds Ratio: {row['Odds_Ratio']:.2f}\n")
                    if not np.isnan(row['Risk_Ratio']):
                        f.write(f"  - Risk Ratio: {row['Risk_Ratio']:.2f}\n")
            
            # Añadir información de correcciones OR/RR
            if df_or_rr_corregidos is not None and not df_or_rr_corregidos.empty:
                f.write("\n" + "="*70 + "\n")
                f.write("CORRECCIONES APLICADAS - OR/RR:\n")
                f.write("="*70 + "\n")
                for _, row in df_or_rr_corregidos.iterrows():
                    f.write(f"✓ {row['Variable']}:\n")
                    f.write(f"  - Risk Ratio corregido: {row['Risk_Ratio_Corregido']:.3f}\n")
                    f.write(f"  - Odds Ratio corregido: {row['Odds_Ratio_Corregido']:.3f}\n")
                    f.write(f"  - Diferencia de riesgo: {row['Diferencia_Riesgo']:.3f}\n")
            
            # Añadir recomendaciones de transformaciones
            if df_transformaciones is not None and not df_transformaciones.empty:
                f.write("\n" + "="*70 + "\n")
                f.write("RECOMENDACIONES DE TRANSFORMACIONES:\n")
                f.write("="*70 + "\n")
                # Encontrar mejor transformación para cada variable
                variables_unicas = df_transformaciones['Variable'].unique()
                for var in variables_unicas:
                    data_var = df_transformaciones[df_transformaciones['Variable'] == var]
                    mejor_transform = data_var.loc[data_var['Asimetría'].abs().idxmin()]
                    f.write(f"✓ {var}: {mejor_transform['Transformación']} (asimetría: {mejor_transform['Asimetría']:.3f})\n")
            
            # Añadir alertas de multicolinealidad
            if correlaciones_problematicas is not None:
                f.write("\n" + "="*70 + "\n")
                f.write("ALERTAS DE MULTICOLINEALIDAD:\n")
                f.write("="*70 + "\n")
                if correlaciones_problematicas:
                    f.write("CORRELACIONES PROBLEMÁTICAS DETECTADAS:\n")
                    for corr in correlaciones_problematicas:
                        f.write(f"⚠ {corr['Variable1']} ↔ {corr['Variable2']}: {corr['Correlación']:.3f}\n")
                    f.write("\nRECOMENDACIÓN: Considerar eliminar una variable de cada par correlacionado\n")
                else:
                    f.write("✅ No se detectaron problemas de multicolinealidad\n")
            
            f.write("\n" + "="*70 + "\n")
            f.write("RECOMENDACIONES PARA ANÁLISIS MULTIVARIABLE:\n")
            f.write("="*70 + "\n")
            
            # Variables recomendadas ordenadas por Cohen's d
            todas_significativas = []
            for _, row in vars_significativas.iterrows():
                todas_significativas.append({
                    'Variable': row['Variable'],
                    'Correlacion': row['Correlacion_Spearman'],
                    'Cohens_d': row['Cohens_d'],
                    'Tamaño_Efecto': row['Tamaño_Efecto']
                })
            
            # Ordenar por valor absoluto de Cohen's d
            todas_significativas.sort(key=lambda x: abs(x['Cohens_d']), reverse=True)
            
            f.write("Variables recomendadas para regresiones lineales multivariables:\n")
            for i, var in enumerate(todas_significativas[:10], 1):
                direccion = "AUMENTA" if var['Correlacion'] > 0 else "DISMINUYE"
                f.write(f"{i}. {var['Variable']}\n")
                f.write(f"   - Correlación: ρ = {var['Correlacion']:.3f} ({direccion})\n")
                f.write(f"   - Tamaño efecto: Cohen's d = {var['Cohens_d']:.3f} ({var['Tamaño_Efecto']})\n")
            
            f.write("\nConsideraciones para el modelo multivariable:\n")
            f.write("- Usar OR/RR corregidos para interpretación\n")
            f.write("- Aplicar transformaciones recomendadas\n")
            f.write("- Evaluar multicolinealidad entre variables predictoras\n")
            f.write("- Considerar interacciones entre variables personales y laborales\n")
            f.write("- Priorizar variables con mayor Cohen's d para mayor poder explicativo\n")
    
    def _crear_metadatos_analisis(self):
        """Crear archivo de metadatos del análisis"""
        with open(f'{self.output_path}/METADATOS_ANALISIS.txt', 'w', encoding='utf-8') as f:
            f.write("METADATOS DEL ANÁLISIS - AUSENTISMO LABORAL\n")
            f.write("="*50 + "\n\n")
            f.write(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total de observaciones: {len(df_clean):,}\n")
            f.write(f"Variables personales analizadas: {len(VARIABLES_PERSONALES)}\n")
            f.write(f"Variables laborales analizadas: {len(VARIABLES_LABORALES)}\n")
            f.write(f"Variables binarias analizadas: {len(VARIABLES_BINARIAS)}\n")
            f.write(f"Umbral de significancia: p < 0.05\n")
            f.write(f"Métodos estadísticos: Spearman, Mann-Whitney, Kruskal-Wallis\n")
            f.write(f"Correcciones aplicadas: OR/RR, transformaciones, multicolinealidad\n")

# Llamada ACTUALIZADA con todos los parámetros nuevos
print("📤 Exportando resultados con correcciones aplicadas...")
exportador = ExportadorResultados(config.OUTPUT_PATH)

# Asegurarse de que las variables nuevas existen (si no, pasar None)
try:
    df_or_rr_corregidos_var = df_or_rr_corregidos if 'df_or_rr_corregidos' in locals() else None
except:
    df_or_rr_corregidos_var = None

try:
    df_transformaciones_var = df_transformaciones if 'df_transformaciones' in locals() else None
except:
    df_transformaciones_var = None

try:
    correlaciones_problematicas_var = correlaciones_problematicas if 'correlaciones_problematicas' in locals() else None
except:
    correlaciones_problematicas_var = None

exportador.exportar_resultados_completos(
    df_results_numeric, 
    df_results_categorical, 
    df_normality, 
    df_clean,
    df_results_binarias_corregido,
    df_or_rr_corregidos_var,        # NUEVO: resultados OR/RR corregidos
    df_transformaciones_var,         # NUEVO: análisis de transformaciones
    correlaciones_problematicas_var  # NUEVO: análisis de multicolinealidad
)

print(f"📁 Resultados exportados en: {config.OUTPUT_PATH}")
print("✓ resultados_numericos.csv")
print("✓ resultados_categoricos.csv") 
print("✓ analisis_binarias_corregido.csv")
print("✓ correcciones_or_rr.csv (NUEVO)")
print("✓ analisis_transformaciones.csv (NUEVO)")
print("✓ multicolinealidad_alertas.txt (NUEVO)")
print("✓ REPORTE_EJECUTIVO_ACTUALIZADO.txt (NUEVO)")
print("✓ METADATOS_ANALISIS.txt (NUEVO)")

In [ ]:
print("\n" + "="*80)
print("ANÁLISIS COMPLETADO - RESUMEN EJECUTIVO")
print("="*80)

# Métricas finales
vars_analizadas = len(VARIABLES_PERSONALES) + len(VARIABLES_LABORALES)  # Excluimos temporales
vars_significativas_numeric = len(df_results_numeric[df_results_numeric['Significativa_Spearman']])
vars_significativas_categorical = df_results_categorical['Significativa_Principal'].sum()
vars_significativas_binarias = len(df_results_binarias_corregido[df_results_binarias_corregido['Significativa']]) if 'df_results_binarias_corregido' in locals() else 0

# Contar variables personales y laborales significativas
vars_personales_significativas = [var for var in VARIABLES_PERSONALES 
                                 if var in df_results_numeric['Variable'].values 
                                 and df_results_numeric[df_results_numeric['Variable'] == var]['Significativa_Spearman'].any()]

vars_laborales_significativas = [var for var in VARIABLES_LABORALES 
                                if var in df_results_numeric['Variable'].values 
                                and df_results_numeric[df_results_numeric['Variable'] == var]['Significativa_Spearman'].any()]

print(f"📊 MÉTRICAS DEL ANÁLISIS:")
print(f"• Variables analizadas (personales + laborales): {vars_analizadas}")
print(f"• Variables personales significativas: {len(vars_personales_significativas)}")
print(f"• Variables laborales significativas: {len(vars_laborales_significativas)}")
print(f"• Variables binarias significativas: {vars_significativas_binarias}")
print(f"• Variables temporales (excluidas): {len(VARIABLES_TEMPORALES)}")
print(f"• Ruta de resultados: {config.OUTPUT_PATH}")

# Distribución de categorías de ausentismo
distribucion_ausentismo = df_clean[TARGET_CATEGORICO].value_counts()
print(f"\n📈 DISTRIBUCIÓN DE CATEGORÍAS DE AUSENTISMO:")
for categoria, count in distribucion_ausentismo.items():
    porcentaje = (count / len(df_clean)) * 100
    print(f"  • {categoria}: {count} empleados ({porcentaje:.1f}%)")

# Variables más importantes (solo significativas personales y laborales)
if not df_results_numeric.empty:
    variables_significativas = df_results_numeric[
        df_results_numeric['Significativa_Spearman'] & 
        (df_results_numeric['Variable'].isin(VARIABLES_PERSONALES + VARIABLES_LABORALES))
    ]
    if not variables_significativas.empty:
        top_variables = variables_significativas.nlargest(5, 'abs_corr')
        print(f"\n🏆 TOP 5 VARIABLES MÁS IMPORTANTES (PERSONALES Y LABORALES):")
        for i, (_, row) in enumerate(top_variables.iterrows(), 1):
            direccion = "Aumenta" if row['Correlacion_Spearman'] > 0 else "Disminuye"
            # Clasificar tipo
            tipo = "PERSONAL" if row['Variable'] in VARIABLES_PERSONALES else "LABORAL"
            print(f"{i}. {row['Variable']} ({tipo}):")
            print(f"   - ρ = {row['Correlacion_Spearman']:.3f} ({row['Fuerza_Spearman']})")
            print(f"   - Cohen's d = {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
            print(f"   - {direccion} ausentismo")

# Interpretación contextual mejorada
def interpretar_resultados_clave(df_results_numeric, df_results_binarias_corregido):
    print(f"\n🔍 HALLAZGOS CLAVE POR TIPO DE VARIABLE:")
    
    print("\n  🎯 VARIABLES PERSONALES SIGNIFICATIVAS:")
    personales_significativas = df_results_numeric[
        df_results_numeric['Significativa_Spearman'] & 
        (df_results_numeric['Variable'].isin(VARIABLES_PERSONALES))
    ]
    
    if not personales_significativas.empty:
        for _, row in personales_significativas.iterrows():
            efecto = "aumenta" if row['Correlacion_Spearman'] > 0 else "disminuye"
            print(f"    • {row['Variable']}: {efecto} el ausentismo")
            print(f"      - Correlación: ρ = {row['Correlacion_Spearman']:.3f}")
            print(f"      - Tamaño efecto: d = {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
    else:
        print("    No hay variables personales significativas")
    
    print("\n  💼 VARIABLES LABORALES SIGNIFICATIVAS:")
    laborales_significativas = df_results_numeric[
        df_results_numeric['Significativa_Spearman'] & 
        (df_results_numeric['Variable'].isin(VARIABLES_LABORALES))
    ]
    
    if not laborales_significativas.empty:
        for _, row in laborales_significativas.iterrows():
            efecto = "aumenta" if row['Correlacion_Spearman'] > 0 else "disminuye"
            print(f"    • {row['Variable']}: {efecto} el ausentismo")
            print(f"      - Correlación: ρ = {row['Correlacion_Spearman']:.3f}")
            print(f"      - Tamaño efecto: d = {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
    else:
        print("    No hay variables laborales significativas")
    
    # Hallazgos binarios CORREGIDOS
    if df_results_binarias_corregido is not None:
        print("\n  🔘 VARIABLES BINARIAS SIGNIFICATIVAS:")
        binarias_significativas = df_results_binarias_corregido[df_results_binarias_corregido['Significativa']]
        if not binarias_significativas.empty:
            for _, row in binarias_significativas.iterrows():
                tipo = "PERSONAL" if row['Variable'] in ['Social_drinker', 'Social_smoker'] else "LABORAL"
                print(f"    • {row['Variable']} ({tipo}): {row['Interpretacion_Direccion']} ausentismo")
                print(f"      - Test: {row['Test_Utilizado']}, p = {row['p_valor']:.4f}")
                print(f"      - Cohen's d: {row['Cohens_d']:.3f} ({row['Tamaño_Efecto']})")
        else:
            print("    No hay variables binarias significativas")

interpretar_resultados_clave(df_results_numeric, df_results_binarias_corregido)

print(f"\n✅ ANÁLISIS COMPLETADO EXITOSAMENTE!")
print(f"📁 Resultados guardados en: {config.OUTPUT_PATH}")
print(f"📄 Informe ejecutivo: {config.OUTPUT_PATH}/INFORME_EJECUTIVO_DETALLADO.txt")
print(f"🔮 Próximo paso: Usar variables significativas para regresiones lineales multivariables")
print(f"💡 Considerar: Variables con Cohen's d > 0.5 tienen efecto mediano/grande")

10. Reporte Final y Métricas

In [ ]:
print("\n" + "="*80)
print("ANÁLISIS COMPLETADO - RESUMEN EJECUTIVO")
print("="*80)

# Métricas finales
vars_analizadas = len(VARIABLES_NUMERICAS) + len(VARIABLES_CATEGORICAS)
vars_significativas_numeric = len(df_results_numeric[df_results_numeric['Significativa_Spearman']])
vars_significativas_categorical = df_results_categorical['Significativa_Principal'].sum()
vars_significativas_binarias = len(df_results_binarias_corregido[df_results_binarias_corregido['Significativa']]) if 'df_results_binarias_corregido' in locals() else 0

# Contar variables personales y laborales significativas
vars_personales_significativas = [var for var in VARIABLES_PERSONALES 
                                 if var in df_results_numeric['Variable'].values 
                                 and df_results_numeric[df_results_numeric['Variable'] == var]['Significativa_Spearman'].any()]

vars_laborales_significativas = [var for var in VARIABLES_LABORALES 
                                if var in df_results_numeric['Variable'].values 
                                and df_results_numeric[df_results_numeric['Variable'] == var]['Significativa_Spearman'].any()]

print(f"📊 MÉTRICAS DEL ANÁLISIS:")
print(f"• Variables analizadas: {vars_analizadas}")
print(f"• Variables numéricas significativas: {vars_significativas_numeric}")
print(f"• Variables categóricas significativas: {vars_significativas_categorical}")
print(f"• Variables binarias significativas: {vars_significativas_binarias}")
print(f"• Variables PERSONALES significativas: {len(vars_personales_significativas)}")
print(f"• Variables LABORALES significativas: {len(vars_laborales_significativas)}")
print(f"• Ruta de resultados: {config.OUTPUT_PATH}")

# Distribución de categorías de ausentismo
distribucion_ausentismo = df_clean[TARGET_CATEGORICO].value_counts()
print(f"\n📈 DISTRIBUCIÓN DE CATEGORÍAS DE AUSENTISMO:")
for categoria, count in distribucion_ausentismo.items():
    porcentaje = (count / len(df_clean)) * 100
    print(f"  • {categoria}: {count} empleados ({porcentaje:.1f}%)")

# Variables más importantes (solo significativas)
if not df_results_numeric.empty:
    variables_significativas = df_results_numeric[df_results_numeric['Significativa_Spearman']]
    if not variables_significativas.empty:
        top_variables = variables_significativas.nlargest(5, 'abs_corr')
        print(f"\n🏆 TOP 5 VARIABLES MÁS IMPORTANTES (SIGNIFICATIVAS):")
        for i, (_, row) in enumerate(top_variables.iterrows(), 1):
            direccion = "Aumenta" if row['Correlacion_Spearman'] > 0 else "Disminuye"
            # Clasificar tipo
            tipo = "PERSONAL" if row['Variable'] in VARIABLES_PERSONALES else "LABORAL" if row['Variable'] in VARIABLES_LABORALES else "TEMPORAL"
            print(f"{i}. {row['Variable']} ({tipo}): ρ = {row['Correlacion_Spearman']:.3f} - {direccion} ausentismo")

# Interpretación contextual mejorada
def interpretar_resultados_clave(df_results_numeric, df_results_binarias_corregido):
    print(f"\n🔍 HALLAZGOS CLAVE POR TIPO DE VARIABLE:")
    
    print("\n  🎯 VARIABLES PERSONALES:")
    personales_claves = {
        'Son': "Empleados con hijos tienen mayor ausentismo",
        'Social_drinker': "Consumo social de alcohol asociado con más ausentismo",
        'Age': "La edad muestra relación con patrones de ausentismo",
        'Body_mass_index': "Índice de masa corporal relacionado con salud y ausentismo"
    }
    
    for var, interpret in personales_claves.items():
        if var in df_results_numeric['Variable'].values:
            resultado = df_results_numeric[df_results_numeric['Variable'] == var].iloc[0]
            if resultado['Significativa_Spearman']:
                print(f"    • {var}: {interpret} (ρ={resultado['Correlacion_Spearman']:.3f})")
    
    print("\n  💼 VARIABLES LABORALES:")
    laborales_claves = {
        'Transportation_expense': "Mayor gasto en transporte asociado con más ausentismo",
        'Disciplinary_failure': "Fallas disciplinarias relacionadas con patrones de ausentismo", 
        'Service_time': "Tiempo de servicio en la empresa afecta ausentismo",
        'Hit_target': "Cumplimiento de metas influye en ausentismo"
    }
    
    for var, interpret in laborales_claves.items():
        if var in df_results_numeric['Variable'].values:
            resultado = df_results_numeric[df_results_numeric['Variable'] == var].iloc[0]
            if resultado['Significativa_Spearman']:
                print(f"    • {var}: {interpret} (ρ={resultado['Correlacion_Spearman']:.3f})")
    
    # Hallazgos binarios CORREGIDOS
    if df_results_binarias_corregido is not None:
        print("\n  🔘 VARIABLES BINARIAS:")
        for _, row in df_results_binarias_corregido[df_results_binarias_corregido['Significativa']].iterrows():
            tipo = "PERSONAL" if row['Variable'] in ['Social_drinker', 'Social_smoker'] else "LABORAL"
            if row['Variable'] == 'Disciplinary_failure':
                print(f"    • {row['Variable']} ({tipo}): Empleados con fallas disciplinarias tienen MENOS ausentismo")
                print(f"      - Diferencia: {row['Media_Grupo1']:.1f} vs {row['Media_Grupo0']:.1f} horas")
            elif row['Variable'] == 'Social_drinker':
                print(f"    • {row['Variable']} ({tipo}): Bebedores sociales tienen MÁS ausentismo")
                print(f"      - Diferencia: {row['Media_Grupo1']:.1f} vs {row['Media_Grupo0']:.1f} horas")

interpretar_resultados_clave(df_results_numeric, df_results_binarias_corregido)

print(f"\n✅ ANÁLISIS COMPLETADO EXITOSAMENTE!")
print(f"📁 Resultados guardados en: {config.OUTPUT_PATH}")
print(f"📄 Informe ejecutivo: {config.OUTPUT_PATH}/INFORME_EJECUTIVO_DETALLADO.txt")
print(f"🔮 Próximo paso: Usar variables significativas para regresiones lineales multivariables")